In [1]:
# Packages not preinstalled on Kaggle that these scripts need.
# GalaxyMNIST is pinned to the commit scripts/train.py's original comments reference,
# for exact reproducibility.
!pip install -q einops torchinfo h5py pygame
!pip install -q "git+https://github.com/mwalmsley/galaxy_mnist.git@c1fe9853a00bc34b2ff082585c6bb1654d34d239"


  Preparing metadata (setup.py) ... done


In [2]:
import os, sys
from pathlib import Path

SMOKE_TEST = False  # True -> overrides EPOCHS to 2 in every script, for a fast pipeline check

REPO_ROOT = Path("/kaggle/working/s4d_repo")
(REPO_ROOT / "model").mkdir(parents=True, exist_ok=True)
(REPO_ROOT / "scripts").mkdir(parents=True, exist_ok=True)
(REPO_ROOT / "logs").mkdir(parents=True, exist_ok=True)

os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))
print("Working directory:", os.getcwd())


Working directory: /kaggle/working/s4d_repo


In [3]:
%%writefile utils.py
"""
Minimal stand-in for `utils.set_pbar_style`, imported by `scripts/train.py`.

`scripts/train.py` does `from utils import set_pbar_style`, but no `utils.py`
was present in the uploaded s4d.zip. Without this stub, importing
`train.py` -- which `train_hybrid.py`, `train_cnn_only.py`, and
`train_hybrid_scale.py` all do, to reuse its `train()` function -- fails
with `ModuleNotFoundError: No module named 'utils'`.

`set_pbar_style` only ever affected tqdm progress-bar colors in the
original notebook-style script body; it has no effect on training logic,
so this no-op-safe stub is a purely cosmetic substitute.
"""


def set_pbar_style(bar_fill_color="#FFFFFF", text_color="#FFFFFF"):
    """Cosmetic no-op. The original styling implementation wasn't in the
    uploaded zip, so this stub just accepts the same call signature used
    in scripts/train.py without changing tqdm's behavior."""
    return None


Writing utils.py


In [4]:
%%writefile kaggle_extras.py
"""
kaggle_extras.py -- not part of the original s4d.zip.

Shared helpers added for the Kaggle notebook run:
  - a torchinfo model summary for every model trained (architecture +
    per-layer param counts), printed and saved alongside each run's
    other artifacts;
  - the same classification metrics the LaTeX report uses throughout its
    master results table (accuracy, precision/recall/F1 macro, one-vs-rest
    ROC-AUC macro), for the two scripts (train_hybrid.py,
    train_hybrid_scale.py) that didn't already compute them;
  - staging of weights, training curves, and confusion matrices into a
    flat, easy-to-find/download location directly under /kaggle/working/,
    instead of nested inside this repo's own working directory.
"""
import glob
import os
import re
import shutil

import numpy as np
import torch
from sklearn.metrics import precision_recall_fscore_support, roc_auc_score
from sklearn.preprocessing import label_binarize

KAGGLE_ROOT = "/kaggle/working"
OUTPUTS_ROOT = os.path.join(KAGGLE_ROOT, "outputs")


def safe_slug(name):
    """Turn a human-readable model name into a filesystem-safe slug."""
    s = re.sub(r"[^\w\-.]+", "_", name.strip())
    return re.sub(r"_+", "_", s).strip("_")


def output_dir(script_tag):
    """/kaggle/working/outputs/<script_tag>/ -- created on first use.

    Falls back to a local ./outputs/<script_tag>/ if /kaggle/working isn't
    writable (e.g. testing outside Kaggle), so a script never crashes
    purely because it isn't running in a Kaggle kernel.
    """
    d = os.path.join(OUTPUTS_ROOT, script_tag)
    try:
        os.makedirs(d, exist_ok=True)
    except OSError:
        d = os.path.join("outputs", script_tag)
        os.makedirs(d, exist_ok=True)
    return d


def print_and_save_summary(model, input_size, name, out_dir):
    """Run torchinfo.summary(), print it, and save the text alongside the
    other artifacts for this run. Falls back to a plain parameter count if
    torchinfo isn't importable for some reason, rather than crashing a
    training run over a reporting nicety."""
    header = f"\n{'=' * 70}\nModel summary: {name}\n{'=' * 70}"
    print(header)
    try:
        from torchinfo import summary as _summary
        stats = _summary(model, input_size=input_size, verbose=0,
                          col_names=("input_size", "output_size", "num_params"))
        text = str(stats)
    except Exception as exc:  # pragma: no cover
        n_params = sum(p.numel() for p in model.parameters())
        text = f"(torchinfo summary unavailable: {exc})\nTotal params: {n_params:,}"
    print(text)
    with open(os.path.join(out_dir, f"{safe_slug(name)}_summary.txt"), "w") as f:
        f.write(header + "\n" + text + "\n")
    return text


def save_weights(model, name, out_dir):
    """Save model.state_dict() so trained weights survive past the Kaggle
    session, not just the in-memory results dict."""
    path = os.path.join(out_dir, f"{safe_slug(name)}_weights.pth")
    torch.save(model.state_dict(), path)
    print(f"Saved weights -> {path}")
    return path


def compute_classification_metrics(all_targets, all_preds, all_probs, class_names):
    """Precision/recall/F1 (macro) + one-vs-rest ROC-AUC (macro) -- the
    same metrics used throughout the LaTeX report's master results table
    (Acc / F1 / Prec / Rec / AUC columns)."""
    labels_arr = list(range(len(class_names)))
    prec_macro, rec_macro, f1_macro, _ = precision_recall_fscore_support(
        all_targets, all_preds, labels=labels_arr, average="macro", zero_division=0
    )
    y_true_bin = label_binarize(all_targets, classes=labels_arr)
    probs_arr = np.array(all_probs)
    try:
        auc_macro = float(roc_auc_score(y_true_bin, probs_arr, average="macro", multi_class="ovr"))
    except ValueError:
        # can happen if a class is entirely absent from the test batch
        auc_macro = float("nan")
    return {
        "precision_macro": float(prec_macro),
        "recall_macro": float(rec_macro),
        "f1_macro": float(f1_macro),
        "roc_auc_macro": auc_macro,
    }


def stage_outputs(out_dir, *patterns):
    """Copy every file matching each glob pattern into out_dir. Copies,
    not moves -- originals stay where the script originally wrote them
    too, in case anything else in this notebook still reads them from
    their original relative path (the results-display cells do)."""
    copied = []
    for pattern in patterns:
        for src in glob.glob(pattern):
            dst = os.path.join(out_dir, os.path.basename(src))
            shutil.copy2(src, dst)
            copied.append(dst)
    print(f"Staged {len(copied)} file(s) -> {out_dir}")
    return copied


Writing kaggle_extras.py


In [5]:
%%writefile model/__init__.py
"""
Galaxy Classification Model Package
"""

from .gclassifier import GalaxyClassifierS4D
from .gclassifier_hybrid import GalaxyClassifierCNNS4D
from .gclassifier_cnn_only import GalaxyClassifierCNNOnly
from .cnn_stem import CNNStem
from . import functions
from .interface import ModelInterface

# NOTE (Kaggle notebook): the interactive pygame GUI is unrelated to
# training and isn't used by any of the four training scripts this
# notebook runs. The import is wrapped defensively so that an unrelated
# GUI/display dependency hiccup in a headless Kaggle kernel can never
# block model training (pygame does import cleanly headless in testing,
# so this is a belt-and-suspenders guard, not a workaround for a known
# failure).
try:
    from .gui import GalaxyExplorerGUI
except Exception as _gui_exc:  # pragma: no cover
    GalaxyExplorerGUI = None
    print(f"[model/__init__] GalaxyExplorerGUI unavailable ({_gui_exc}); "
          f"not needed for training, continuing.")

__all__ = [
    'GalaxyClassifierS4D',
    'GalaxyClassifierCNNS4D',
    'GalaxyClassifierCNNOnly',
    'CNNStem',
    'functions',
    'ModelInterface',
    'GalaxyExplorerGUI',
]


Writing model/__init__.py


In [6]:
%%writefile model/tlts.py
import torch
import torch.nn as nn

class TakeLastTimestep(nn.Module):
    """
    Module that extracts the last timestep from a sequence.

    This layer is used to summarize sequence outputs from recurrent 
    or sequence models by taking only the final timestep as a feature vector.

    Parameters
    ----------
    None

    Input
    -----
    x : torch.Tensor
        Input tensor of shape (B, L, D), where
        B : batch size,
        L : sequence length,
        D : feature dimension.

    Returns
    -------
    out : torch.Tensor
        Output tensor of shape (B, D), corresponding to the last timestep
        of each sequence in the batch.
    """
    def forward(self, x):
        # x: (B, L, D)
        # FIX (Kaggle notebook, applied on top of the uploaded zip): this
        # method was `return x.mean(dim=1)` -- mean-pooling, not
        # last-timestep pooling. That contradicted this class's own name
        # and docstring, its own commented-out self-test below (which
        # asserts the output equals x[:, -1, :]), and the reference
        # `TakeLastTimestep` in notebook-best-s4d-model_1_.ipynb, which
        # correctly implements last-timestep pooling. Every S4D model in
        # this repo (GalaxyClassifierS4D, GalaxyClassifierCNNS4D, and the
        # grid model in train_ablation.py) uses this class unconditionally,
        # so as shipped every "last pooling" run was silently mean-pooling.
        # Restored to genuine last-timestep pooling to match both the
        # documented intent and the reference notebook.
        return x[:, -1, :]

"""
if __name__ == "__main__":
    print("Testing tlts code")

    layer = TakeLastTimestep()
    print("Layer established")

    x = torch.randn(3, 6, 2)
    print(f"Input shape: {x.shape}")

    output = layer(x)
    print(f"Output shape: {output.shape}")
    print(f"Expected (3, 2): {output.shape == (3, 2)}")
    print(f"Match? {torch.allclose(x[0, -1, :], output[0, :])}")

    print("Successful")
"""
"""
Explaination:
The TakeLastTimeStep layer transforms an input tensor of shape (B, L, D) 
into an output tensor of shape (B, D) by indexing the last position.
The hidden state at position L has been updated by all L previous inputs
and by the time model reaches position L-1, which is the last time step, 
the state has came across and collected information from all inputs u(0) 
through u(L-1) and therefore the (B, D) tensor at the final position
serves as a compressed summary of the entire (B, L, D) sequence.
This is identical to how RNNs, LSTMs, and GRUs use their final hidden state
for classification tasks — the last timestep naturally accumulates the 
history of the whole sequence through the recurrent processes.
"""


Writing model/tlts.py


In [7]:
%%writefile model/hilbert.py
import torch   
import torch.nn as nn


class HilbertScan(nn.Module):
    """
    Reorders pixels according to a Hilbert Curve for multi-channel images.
    
    The Hilbert curve is a space-filling curve that preserves spatial locality
    when mapping 2D coordinates to 1D sequences. This module applies the same
    Hilbert curve pattern to each channel independently, then reorganizes the
    output so the sequence dimension comes first.
    
    Supports grayscale (C=1) or RGB (C=3) images.
    
    Attributes
    ----------
    indices : torch.LongTensor
        Precomputed Hilbert curve indices for an n×n grid, stored as a
        non-trainable buffer.
    
    Input
    -----
    x : torch.Tensor
        Input tensor of shape (B, C, H, W), where
        B : batch size
        C : number of channels
        H : height (n)
        W : width (n)
    
    Returns
    -------
    out : torch.Tensor
        Reordered tensor of shape (B, seq_len, C) where seq_len = H*W = n*n.
        Pixels are arranged according to the Hilbert curve traversal order.
    """
    def __init__(self, n=64):
        """Initialize HilbertScan with precomputed indices for an n x n grid.

        Parameters
        ----------
        n : int, optional
            Grid size (must be a power of 2). Default 64, which keeps the
            existing GalaxyClassifierS4D baseline (scanning the raw 64x64
            image) unaffected. The hybrid CNN+S4D classifier passes a
            smaller n (e.g. 16) since it scans the CNN stem's downsampled
            feature map instead of the raw image.
        """
        super().__init__()
        self.n = n
        indices = self.get_hilbert_indices(n)
        self.register_buffer('indices', indices)

    def _rot(self, s, x, y, rx, ry):
    
        if ry == 0:                  # Bottom half of the current square
            if rx == 1:              # Bottom-right quadrant
                x = s - 1 - x        # Reflect over diagonal
                y = s - 1 - y
            x, y = y, x              # Swap x and y for 90° rotation
        return x, y


    def _d2xy(self, n, d):
        """
        Convert 1D Hilbert curve distance to 2D coordinates.
        
        This implements the Hilbert curve mapping algorithm that converts
        a linear distance along the curve to (x, y) coordinates.
        
        Parameters
        ----------
        n : int
            Size of the grid (must be a power of 2).
        d : int
            Distance along the Hilbert curve (0 to n²-1).
        
        Returns
        -------
        tuple of int
            (x, y) coordinates in the grid.
        """
        x = 0
        y = 0
        t = d
        s = 1
#Determine which quadrant of the current square this distance is in
        while s < n:
            rx = (t // 2) & 1
            ry = (t ^ rx) & 1
#Rotate and/or reflect coordinates depending on quadrant
            x, y = self._rot(s, x, y, rx, ry)
            x += s * rx
            y += s * ry
#Move to next level of recursion (divide distance by 4 for next smaller square)
            t //= 4
            s *= 2

        return x, y

    def get_hilbert_indices(self, n):
        """
        Generate Hilbert curve indices for an n x n grid.
        
        Creates a lookup table that maps Hilbert curve positions to
        flattened array indices for a 2D grid.
        
        Parameters
        ----------
        n : int
            Grid size (must be a power of 2).
        
        Returns
        -------
        torch.LongTensor
            Tensor of shape (n²,) containing flattened indices following
            the Hilbert curve traversal order.
        """
        indices = []
        for d in range(n * n):
            x, y = self._d2xy(n, d)
            # GalaxyMNIST is 64x64, power of 2
            if x < n and y < n:
                indices.append(y * n + x)
        return torch.LongTensor(indices)

    def forward(self, x):
        """
        Apply Hilbert curve reordering to input images.
        
        Parameters
        ----------
        x : torch.Tensor
            Input images of shape (B, C, H, W).
        
        Returns
        -------
        torch.Tensor
            Reordered tensor of shape (B, seq_len, C) where seq_len = H*W,
            with pixels arranged in Hilbert curve order.
        """
        # x: (B, C, H, W)
        B, C, H, W = x.shape
        x = x.view(B, C, -1)           # Flatten each channel: (B, C, H*W)
        x = x[:, :, self.indices]      # Reorder according to Hilbert: (B, C, H*W)
        x = x.permute(0, 2, 1)         # (B, seq_len, C) so sequence dimension is 1D
        return x
if __name__ == "__main__":
    import torch

    img = torch.arange(64*64).view(1,1,64,64).float()
    hilbert = HilbertScan()
    out = hilbert(img)

    print(out[0, :20, 0])

Writing model/hilbert.py


In [8]:
%%writefile model/cnn_stem.py
import torch
import torch.nn as nn


class CNNStem(nn.Module):
    """
    Convolutional stem for the CNN-stem -> S4D hybrid classifier.

    RESEARCH VERSION (post-course). The course version of this stem was
    constrained to plain Conv+GELU (no BatchNorm/LayerNorm) for eventual
    bare-metal RISC-V portability. That constraint is dropped here since
    we're now optimizing purely for accuracy -- if a bare-metal export is
    ever needed again, GroupNorm's running stats can be folded into the
    preceding conv's weights at export time (standard conv-BN/GN fusion),
    so this doesn't have to be a permanent trade-off even for that goal.

    Key differences from the course version:
      1. A stride-1 "detail" conv runs FIRST, at full input resolution,
         before any downsampling happens. The old stem's first conv was
         already stride-2, so it only ever saw a raw 3x3 window of
         un-processed pixels before halving resolution -- thin, low-
         contrast structures (e.g. dust lanes in edge-on spirals, which
         is exactly the signal needed to separate Smooth Cigar from
         Edge-on Disk) had no chance to be extracted before being pooled
         away. Now there's a full-res feature-extraction pass first.
      2. GroupNorm after every conv (stable training, no batch-size
         dependence, no running stats to worry about -- unlike
         BatchNorm, GroupNorm's stats are computed per-sample so it
         behaves identically in train/eval).
      3. More channel capacity (mid_channels default raised 16 -> 32).
      4. Residual add on the stride-1 block (cheap, helps optimization,
         doesn't change spatial dims so it's a free add).

    Two variants, selected via `reduction` (same semantics as before):
      - reduction=16: three conv stages, 64x64 -> 64x64 (stride1) ->
        32x32 -> 16x16   => 16x sequence-length cut (4096->256)
      - reduction=4:  64x64 -> 64x64 (stride1) -> 32x32
                                                    => 4x cut (4096->1024)

    Parameters
    ----------
    in_channels : int
        Number of input image channels (1 grayscale, 3 RGB). Use 3 --
        color carries the dust-lane / reddening signal that grayscale
        (channel-averaged) input throws away.
    d_model : int, optional
        Output channel count of the stem, feeding S4D's d_model. Default 64.
    mid_channels : int, optional
        Hidden channel width of the stem's early conv stages. Default 32
        (was 16 in the course version -- more capacity now that accuracy,
        not param-count / embedded footprint, is the objective).
    reduction : int, optional
        Spatial / sequence-length reduction factor. One of {4, 16}.
        Default 16.
    dropout : float, optional
        Spatial dropout (Dropout2d) applied after the stride-1 block, as
        light regularization for the ~8k-image dataset. Default 0.1.

    Input
    -----
    x : torch.Tensor, shape (B, in_channels, 64, 64)

    Returns
    -------
    torch.Tensor
        shape (B, d_model, 16, 16) if reduction=16,
        shape (B, d_model, 32, 32) if reduction=4.
    """

    def __init__(self, in_channels, d_model=64, mid_channels=32, reduction=16, dropout=0.1):
        super().__init__()
        if reduction not in (4, 16):
            raise ValueError(f"reduction must be 4 or 16, got {reduction}")
        self.reduction = reduction

        def gn(channels):
            # GroupNorm needs num_groups | channels; 8 groups is a safe
            # default for the channel counts used here (32, 64).
            groups = 8 if channels % 8 == 0 else 1
            return nn.GroupNorm(groups, channels)

        # --- Stage 0: full-resolution detail extraction (stride 1) ---
        # This is the change that matters most: features are computed at
        # the input's native 64x64 resolution before anything is thrown
        # away, so thin/low-contrast structures (dust lanes, arm edges)
        # actually get a chance to be represented.
        self.stem_conv = nn.Conv2d(in_channels, mid_channels, kernel_size=3, stride=1, padding=1)
        self.stem_norm = gn(mid_channels)
        self.stem_act = nn.GELU()

        self.res_conv = nn.Conv2d(mid_channels, mid_channels, kernel_size=3, stride=1, padding=1)
        self.res_norm = gn(mid_channels)
        self.res_act = nn.GELU()
        self.drop = nn.Dropout2d(dropout)

        # --- Downsampling stages ---
        if reduction == 16:
            # 64 -> 32 -> 16
            self.down1 = nn.Conv2d(mid_channels, mid_channels, kernel_size=3, stride=2, padding=1)
            self.down1_norm = gn(mid_channels)
            self.down1_act = nn.GELU()

            self.down2 = nn.Conv2d(mid_channels, d_model, kernel_size=3, stride=2, padding=1)
            self.down2_norm = gn(d_model)
            self.down2_act = nn.GELU()
        else:
            # 64 -> 32
            self.down1 = nn.Conv2d(mid_channels, d_model, kernel_size=3, stride=2, padding=1)
            self.down1_norm = gn(d_model)
            self.down1_act = nn.GELU()
            self.down2 = None

    def forward(self, x):
        # x: (B, in_channels, 64, 64)
        x = self.stem_act(self.stem_norm(self.stem_conv(x)))       # full-res feature extraction
        r = self.res_act(self.res_norm(self.res_conv(x)))
        x = x + r                                                   # residual, still full-res
        x = self.drop(x)

        x = self.down1_act(self.down1_norm(self.down1(x)))
        if self.down2 is not None:
            x = self.down2_act(self.down2_norm(self.down2(x)))
        return x


Writing model/cnn_stem.py


In [9]:
%%writefile model/s4d_recurrent.py
import math
import torch
import torch.nn as nn
from einops import repeat

class S4D(nn.Module):
    """
    Diagonal Structured State Space (S4D) layer.
    
    Implements the S4D variant of Structured State Spaces using diagonal state matrices
    for computational efficiency. This layer processes sequences through a continuous-time
    state space model discretized using the bilinear method, enabling modeling of long-range
    dependencies with linear complexity.
    
    The S4D model parameterizes the state space with:
    - Diagonal complex-valued state transition matrix A
    - Complex-valued output projection matrix C  
    - Skip connection parameter D
    - Learnable discretization timestep dt
    
    Convolution is performed efficiently in the frequency domain using FFT.
    
    Parameters
    ----------
    d_model : int
        Input and output feature dimension (number of independent SSM copies).
    d_state : int, optional
        Latent state dimension (must be even for complex representation). 
        Default is 64.
    dt_min : float, optional
        Minimum discretization timestep. Default is 0.001.
    dt_max : float, optional
        Maximum discretization timestep. Default is 0.1.
    transposed : bool, optional
        If True, expects input shape (B, H, L). If False, expects (B, L, H).
        Default is True.
    lr : float, optional
        Custom learning rate for SSM parameters. If None, uses optimizer default.
        If 0.0, parameters become fixed buffers.
    
    Attributes
    ----------
    h : int
        Number of independent SSM copies (equals d_model).
    n : int
        State dimension.
    log_dt : nn.Parameter or buffer
        Log-space discretization timestep (shape: h).
    log_A_real : nn.Parameter or buffer
        Log-space real part of diagonal state matrix (shape: h, n//2).
    A_imag : nn.Parameter or buffer
        Imaginary part of diagonal state matrix (shape: h, n//2).
    C : nn.Parameter
        Complex output projection matrix (shape: h, n//2, 2 for real view).
    D : nn.Parameter  
        Skip connection weights (shape: h).
    
    Input
    -----
    u : torch.Tensor
        Input sequence of shape (B, H, L) if transposed=True, or (B, L, H) otherwise.
        B : batch size
        H : d_model (feature dimension)
        L : sequence length
    
    Returns
    -------
    y : torch.Tensor
        Output sequence of same shape as input.
    None
        Placeholder for compatibility with stateful interfaces.
    
    References
    ----------
    Gu, A., Goel, K., & Ré, C. (2022). Efficiently Modeling Long Sequences with 
    Structured State Spaces. In ICLR 2022.
    
    Gu, A., Gupta, A., Goel, K., & Ré, C. (2022). On the Parameterization and 
    Initialization of Diagonal State Space Models. In NeurIPS 2022.
    """
    def __init__(self, d_model, d_state=64, dt_min=0.001, dt_max=0.1, transposed=True, lr=None):
        super().__init__()
        self.h = d_model
        self.n = d_state
        self.transposed = transposed

        # --- Initial Parameter Tensors ---
        log_dt = torch.rand(self.h) * (math.log(dt_max) - math.log(dt_min)) + math.log(dt_min)
        log_A_real = torch.log(0.5 * torch.ones(self.h, self.n // 2))
        A_imag = math.pi * repeat(torch.arange(self.n // 2), 'n -> h n', h=self.h)
        C_init = torch.randn(self.h, self.n // 2, dtype=torch.cfloat)

        # --- Registration ---
        # We use 'register' to set weight_decay=0.0 and custom LRs for SSM cores
        self.register("log_dt", log_dt, lr)
        self.register("log_A_real", log_A_real, lr)
        self.register("A_imag", A_imag, lr)
        
        # C and D are usually treated as standard parameters
        self.C = nn.Parameter(torch.view_as_real(C_init))
        self.D = nn.Parameter(torch.randn(self.h))

    def register(self, name, tensor, lr=None):
        """
        Register a parameter or buffer with custom optimization settings.
        
        Parameters
        ----------
        name : str
            Name for the parameter/buffer.
        tensor : torch.Tensor
            Tensor to register.
        lr : float, optional
            Custom learning rate. If 0.0, registers as buffer (non-trainable).
            If None, uses optimizer default. Otherwise, attaches custom lr metadata.
        """
        if lr == 0.0:
            self.register_buffer(name, tensor)
        else:
            self.register_parameter(name, nn.Parameter(tensor))
            # Tag the parameter with optimization constraints
            optim = {"weight_decay": 0.0}
            if lr is not None: 
                optim["lr"] = lr
            setattr(getattr(self, name), "_optim", optim)

    def forward(self, u):
        """
        Forward pass through the S4D layer.
        
        Computes the convolution of the input sequence with the SSM kernel using FFT.
        The kernel is generated from the continuous-time SSM parameters and discretized
        using the learned timestep dt.
        
        Process:
        1. Materialize SSM parameters (dt, A, C) from log-space representations
        2. Generate discrete convolution kernel K via truncated power series
        3. Perform FFT-based convolution: y = K * u
        4. Add skip connection: y = y + D * u
        
        Parameters
        ----------
        u : torch.Tensor
            Input sequence of shape (B, H, L) if transposed=True, else (B, L, H).
            B : batch size
            H : feature dimension (d_model)
            L : sequence length
        
        Returns
        -------
        y : torch.Tensor
            Output sequence of same shape as input.
        None
            Placeholder for state (included for interface compatibility).
        """
        if not self.transposed: u = u.transpose(-1, -2)
        L = u.size(-1)

        # 1. Materialize Parameters
        dt = torch.exp(self.log_dt) 
        C = torch.view_as_complex(self.C) 
        A = -torch.exp(self.log_A_real) + 1j * self.A_imag 

        # 2. Generate Kernel K (Diagonal SSM formula)
        dtA = A * dt.unsqueeze(-1)  
        # Power series generation: exp(A * dt * t)
        K_exp = torch.exp(dtA.unsqueeze(-1) * torch.arange(L, device=u.device)) 
        C_tilde = C * (torch.exp(dtA) - 1.) / A
        k = 2 * torch.einsum('hn, hnl -> hl', C_tilde, K_exp).real 

        # 3. FFT Convolution (y = k * u)
        k_f = torch.fft.rfft(k, n=2*L) 
        u_f = torch.fft.rfft(u, n=2*L) 
        y = torch.fft.irfft(u_f * k_f, n=2*L)[..., :L] 

        # 4. Skip Connection
        y = y + u * self.D.unsqueeze(-1)

        if not self.transposed: y = y.transpose(-1, -2)
        return y, None

Writing model/s4d_recurrent.py


In [10]:
%%writefile model/gclassifier.py
import torch
import torch.nn as nn

from .hilbert import HilbertScan
from .tlts import TakeLastTimestep
from .s4d_recurrent import S4D

class GalaxyClassifierS4D(nn.Module):
    """
    Galaxy classifier using Hilbert Scan and S4 sequence modeling.
    
    This model scans 2D galaxy images into a 1D Hilbert sequence, projects
    the multi-channel pixel values to a higher-dimensional feature space,
    processes the sequence with stacked S4 layers with GELU activations, 
    takes the final timestep as a summary representation, and applies a 
    linear classifier to predict galaxy types.
    
    Parameters
    ----------
    s4_state : int, optional
        Hidden state dimension for the S4 layers (default is 64).
    d_model : int, optional
        Output feature dimension of the S4 layers (default is 64).
    num_classes : int, optional
        Number of output classes (default is 4).
    colored : bool, optional
        If True, expects RGB input images (3 channels); if False, expects
        grayscale images (1 channel) (default is True).
    
    Attributes
    ----------
    seq_len : int
        Sequence length after Hilbert scan (64*64 = 4096).
    d_model : int
        Dimension of the S4 output features.
    hilbert_channels : int
        Number of input channels (1 for grayscale, 3 for RGB).
    hilbert_scan : HilbertScan
        Layer that converts 2D images into 1D sequences using a Hilbert scan.
    uproject : nn.Linear
        Linear projection mapping hilbert_channels to d_model dimensions.
    s4_1 : S4D
        First S4 layer.
    act1 : nn.GELU
        GELU activation after the first S4 layer.
    s4_2 : S4D
        Second S4 layer.
    act2 : nn.GELU
        GELU activation after the second S4 layer.
    take_last : TakeLastTimestep
        Layer that extracts the last timestep from the sequence.
    fc : nn.Linear
        Linear classifier mapping S4 features to output classes.
    softmax : nn.Softmax
        Softmax layer for output probabilities.
    """
    def __init__(self, s4_state=64, d_model=64, num_classes=4, colored=True):
        super().__init__()
        self.seq_len = 64 * 64 
        self.d_model = d_model

        # Hilbert Scan layer
        self.hilbert_scan = HilbertScan()
        self.hilbert_channels = 1 if not colored else 3

        self.uproject = nn.Linear(self.hilbert_channels, d_model)

        # S4 layers -- recurrent, not the old FFT/causal-conv layer. Verified
        # against the trained portable first (see recurrent_vs_causal_conv_verification.png):
        # logits matched to ~6e-4, same argmax on every sample. Conv layer's gone now.
        self.s4_1 = S4D(d_model=d_model, d_state=s4_state, transposed=False)
        self.act1 = nn.GELU()

        self.s4_2 = S4D(d_model=d_model, d_state=s4_state, transposed=False)
        self.act2 = nn.GELU()

        # Take last timestep
        self.take_last = TakeLastTimestep()

        # Classifier
        self.fc = nn.Linear(d_model, num_classes)

        # Softmax for output probabilities
        self.softmax = nn.Softmax(dim=-1)


       # -------------------------------------------------------------------------
        # PARAMETER COUNT VERIFICATION (Task 8.4)
        # Verified with torchinfo.summary()
        # -------------------------------------------------------------------------
        # 1. Input Projection: (1 * 64) + 64 = 128 params
        # 2. S4D Layer 1 (Optimized N/2 symmetry): 
        #    Per feature: 130 params (vs 258 naive)
        #    Total: 64 * 130 = 8,320 params
        # 3. S4D Layer 2: Same as Layer 1 = 8,320 params
        # 4. Classifier Head: (64 * 4) + 4 = 260 params
        # 
        # GRAND TOTAL: 128 + 8,320 + 8,320 + 260 = 17,028 Parameters
        # -------------------------------------------------------------------------



        # -------------------------------------------------------------------------
        # FLOPS ESTIMATION (Task 8.5) -- redone for the recurrent S4D layer
        # Sequence Length L = 4096, d_model = 64, d_state = 64, C = 1
        # -------------------------------------------------------------------------
        # 1. Input Projection: L * C * d_model
        #    4096 * 1 * 64 = 262,144 Ops
        #
        # 2. S4D Layers (x2): no more FFT kernel, so no log(L) term. Each layer
        #    steps through L timesteps, and at each step does ~2 complex MACs per
        #    state element (one for the state update, one for the output sum) --
        #    a complex MAC costs roughly 4x a real one, call it ~8 real ops:
        #    2 * (L * (d_state/2) * d_model * 8) = 2 * (4096*32*64*8) ≈ 134.2M Ops
        #
        # 3. Classifier Head: d_model * Classes
        #    64 * 4 = 256 Ops
        #
        # GRAND TOTAL: ~134.5 Million Operations per forward pass
        # (vs. ~6.55M under the old FFT estimate -- more raw arithmetic, since
        # we lost the O(log L) speedup, but no transcendental-heavy kernel
        # generation either, which is most of why it still benchmarks faster
        # in practice at this d_model -- see model/s4d_recurrent.py)
        # -------------------------------------------------------------------------

    def forward(self, x, return_logits=False):
        """
        Forward pass of the PixelS4Galaxy model.
        
        Parameters
        ----------
        x : torch.Tensor
            Input tensor of shape (B, C, 64, 64), where B is the batch size
            and C is the number of channels (1 for grayscale, 3 for RGB).
        return_logits : bool, optional
            If True, returns raw logits instead of softmax probabilities 
            (default is False).
        
        Returns
        -------
        output : torch.Tensor
            If return_logits=True: Output logits of shape (B, num_classes),
            representing unnormalized scores for each galaxy class.
            If return_logits=False: Output probabilities of shape (B, num_classes),
            representing the softmax probability distribution over classes.
        """
        B, C, H, W = x.shape
        assert H == 64 and W == 64, "Expected 64x64"
        assert C == self.hilbert_channels, f"Expected {self.hilbert_channels} channels"

        # 1. Hilbert scan: 2D > 1D
        x_seq = self.hilbert_scan(x)  # (B,4096,C)

        # 2. Input projection: C > d_model
        x_proj = self.uproject(x_seq)  # (B,4096,d_model)

        # 3. S4D layer 1 + GELU
        s4_out1, _ = self.s4_1(x_proj)
        a1 = self.act1(s4_out1)  # (B,4096,d_model

        # 4. S4D layer 2 + GELU
        s4_out2, _ = self.s4_2(a1)
        a2 = self.act2(s4_out2)      # (B,4096,d_model)

        # 5. Take last timestep
        last = self.take_last(a2)          # (B,d_model)

        # 6. Classifier: d_model > num_classes
        logits = self.fc(last)             # (B,4)

        # Return logits or softmax
        if return_logits:
            return logits
        return self.softmax(logits)

# basically this function takes the image and turns it into a sequence
        # first we check the shape to make sure its 64x64
        # then the hilbert scan flattens the 2D image into a long 1D list of pixels
        # after that we project it up to hidden size using a linear layer
        # then it goes through two S4 layers with GELU activation in between to learn features
        # since its a sequence model we only care about the very last timestep which has the summary
        # finally we pass that last step to the linear classifier to get the 4 class scores
        # and if we need probs we apply softmax otherwise just return the raw logits

        #raise NotImplementedError("Forward method not implemented yet.")

Writing model/gclassifier.py


In [11]:
%%writefile model/gclassifier_hybrid.py
import torch
import torch.nn as nn

from .cnn_stem import CNNStem
from .hilbert import HilbertScan
from .tlts import TakeLastTimestep
from .s4d_recurrent import S4D


class GalaxyClassifierCNNS4D(nn.Module):
    """
    CNN-stem -> S4D hybrid galaxy classifier.

    Companion/competitor to GalaxyClassifierS4D (model/gclassifier.py). The
    baseline assumes long-range pixel dependency matters for galaxy
    morphology (it scans the full 4096-pixel image straight into S4D). This
    model tests the opposite hypothesis: that morphology is dominated by
    local structure (arm curvature, edge sharpness, blob shape), so a small
    CNN stem can do local feature extraction + spatial downsampling first,
    handing S4D a much shorter sequence, while preserving accuracy.

    image (B,C,64,64)
      -> CNNStem                          -> (B, d_model, grid, grid)
      -> HilbertScan(n=grid)              -> (B, grid*grid, d_model)
      -> S4D(d_model, d_state) s4_1       -> (B, grid*grid, d_model)
      -> GELU
      -> S4D(d_model, d_state) s4_2       -> (B, grid*grid, d_model)
      -> GELU
      -> TakeLastTimestep                 -> (B, d_model)
      -> Linear(d_model, num_classes) fc  -> (B, num_classes)
      -> softmax (or raw logits, matching GalaxyClassifierS4D's API exactly)

    With the default stem_reduction=16, grid=16, so seq_len=256 -- a 16x cut
    from the baseline's 4096, and therefore ~16x fewer S4D-loop ops (S4D's
    per-layer cost is O(d_model * seq_len * d_state/2), linear in seq_len --
    see the FLOPS comment in model/gclassifier.py for the exact op-count
    formula this scales).

    The CNN stem's last conv already projects channels up to d_model, so --
    unlike GalaxyClassifierS4D -- there is no separate `uproject` Linear
    here; the stem's output channel dim *is* the projection.

    S4D itself (model/s4d_recurrent.py) is reused unmodified apart from an
    optional 3rd stacked layer (num_s4_layers=3); d_state is still
    configurable via s4_state, only seq_len shrinks because of what
    happens upstream in the stem.

    Parameters
    ----------
    s4_state : int, optional
        Hidden state dimension for the S4D layers (default 64).
    d_model : int, optional
        Output feature dimension of the CNN stem / S4D layers (default 64).
    num_classes : int, optional
        Number of output classes (default 4).
    colored : bool, optional
        If True, expects RGB input images (3 channels); if False, expects
        grayscale images (1 channel). Default True -- color carries the
        dust-lane/reddening signal needed to separate Smooth Cigar from
        Edge-on Disk, the dominant error mode observed with grayscale-only
        input.
    stem_reduction : int, optional
        Sequence-length reduction factor applied by the CNN stem before
        Hilbert-scanning, one of {4, 16}:
          - 16 (default): three-stage stem (stride1 -> stride2 -> stride2),
            64x64 -> 16x16, seq_len 4096 -> 256.
          - 4: milder cut, two-stage stem (stride1 -> stride2),
            64x64 -> 32x32, seq_len 4096 -> 1024. Retains more spatial
            resolution; the recommended default for research runs where
            accuracy matters more than compute savings.
    mid_channels : int, optional
        Hidden channel width inside the stem's stride-1 detail-extraction
        stage. Default 32 (raised from the course version's 16 for more
        capacity).
    stem_dropout : float, optional
        Dropout2d applied inside the stem after the stride-1 block.
        Default 0.1.
    head_dropout : float, optional
        Dropout applied to the pooled sequence representation right before
        the classifier head. Default 0.2.
    num_s4_layers : int, optional
        Number of stacked S4D layers, one of {2, 3}. Default 2 (matches
        the course architecture); set to 3 to test whether 2 layers is a
        capacity bottleneck once the stem/data changes above are in place.

    Attributes
    ----------
    seq_len : int
        Sequence length after the CNN stem + Hilbert scan (256 for
        stem_reduction=16, 1024 for stem_reduction=4).
    d_model : int
        Dimension of the S4D output features.
    hilbert_channels : int
        Number of input image channels (1 for grayscale, 3 for RGB).
    cnn_stem : CNNStem
        Conv stem doing local feature extraction, downsampling, and
        channel projection to d_model.
    hilbert_scan : HilbertScan
        Scans the stem's (B, d_model, grid, grid) feature map into a 1D
        sequence via a Hilbert curve over the grid.
    s4_1, s4_2 : S4D
        Stacked S4D layers (unmodified from the baseline).
    act1, act2 : nn.GELU
    take_last : TakeLastTimestep
    fc : nn.Linear
    softmax : nn.Softmax
    """

    def __init__(self, s4_state=64, d_model=64, num_classes=4, colored=True,
                 stem_reduction=16, mid_channels=32, stem_dropout=0.1,
                 head_dropout=0.2, num_s4_layers=2):
        super().__init__()
        if stem_reduction not in (4, 16):
            raise ValueError(f"stem_reduction must be 4 or 16, got {stem_reduction}")
        if num_s4_layers not in (2, 3):
            raise ValueError(f"num_s4_layers must be 2 or 3, got {num_s4_layers}")

        self.hilbert_channels = 1 if not colored else 3
        self.d_model = d_model
        self.stem_reduction = stem_reduction
        self.num_s4_layers = num_s4_layers

        # Spatial side of the feature grid after the stem: 64 -> 64/sqrt(reduction)
        # reduction=16 -> two stride-2 blocks -> /4 side reduction -> grid=16
        # reduction=4  -> one stride-2 block   -> /2 side reduction -> grid=32
        grid = 64 // (4 if stem_reduction == 16 else 2)
        self.seq_len = grid * grid

        # CNN stem: local feature extraction + downsampling. Its last conv
        # projects channels to d_model, so no separate uproject Linear is
        # needed (unlike the baseline). Research-version stem (see
        # cnn_stem.py) runs a full-resolution stride-1 pass before any
        # downsampling, so thin structures (dust lanes etc.) survive.
        self.cnn_stem = CNNStem(
            in_channels=self.hilbert_channels,
            d_model=d_model,
            mid_channels=mid_channels,
            reduction=stem_reduction,
            dropout=stem_dropout,
        )

        # Hilbert scan over the downsampled feature grid (not the raw image)
        self.hilbert_scan = HilbertScan(n=grid)

        # S4D layers. Stack of 2 (course default) or 3 (research option --
        # a bit more sequence-modeling capacity now that d_model/mid_channels
        # are also bigger, in case 2 layers is now the bottleneck).
        self.s4_1 = S4D(d_model=d_model, d_state=s4_state, transposed=False)
        self.act1 = nn.GELU()

        self.s4_2 = S4D(d_model=d_model, d_state=s4_state, transposed=False)
        self.act2 = nn.GELU()

        if num_s4_layers == 3:
            self.s4_3 = S4D(d_model=d_model, d_state=s4_state, transposed=False)
            self.act3 = nn.GELU()
        else:
            self.s4_3 = None
            self.act3 = None

        # Take last timestep (currently mean-pooling, see tlts.py)
        self.take_last = TakeLastTimestep()

        # Light dropout before the classifier head -- regularization for
        # the ~8k-image dataset now that capacity has gone up.
        self.head_drop = nn.Dropout(head_dropout)

        # Classifier
        self.fc = nn.Linear(d_model, num_classes)

        # Softmax for output probabilities
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, x, return_logits=False):
        """
        Forward pass of the CNN-stem -> S4D hybrid model.

        Parameters
        ----------
        x : torch.Tensor
            Input tensor of shape (B, C, 64, 64), where B is the batch size
            and C is the number of channels (1 for grayscale, 3 for RGB).
        return_logits : bool, optional
            If True, returns raw logits instead of softmax probabilities
            (default is False).

        Returns
        -------
        output : torch.Tensor
            If return_logits=True: Output logits of shape (B, num_classes).
            If return_logits=False: Output probabilities of shape
            (B, num_classes), the softmax distribution over classes.
        """
        B, C, H, W = x.shape
        assert H == 64 and W == 64, "Expected 64x64"
        assert C == self.hilbert_channels, f"Expected {self.hilbert_channels} channels"

        # 1. CNN stem: local feature extraction + spatial downsampling,
        #    also projects channels -> d_model
        feat = self.cnn_stem(x)  # (B, d_model, grid, grid)

        # 2. Hilbert scan: 2D feature map -> 1D sequence
        x_seq = self.hilbert_scan(feat)  # (B, seq_len, d_model)

        # 3. S4D layer 1 + GELU
        s4_out1, _ = self.s4_1(x_seq)
        a1 = self.act1(s4_out1)  # (B, seq_len, d_model)

        # 4. S4D layer 2 + GELU
        s4_out2, _ = self.s4_2(a1)
        a2 = self.act2(s4_out2)  # (B, seq_len, d_model)

        # 4b. Optional S4D layer 3 + GELU (num_s4_layers=3)
        if self.s4_3 is not None:
            s4_out3, _ = self.s4_3(a2)
            a2 = self.act3(s4_out3)  # (B, seq_len, d_model)

        # 5. Take last timestep (mean-pool, see tlts.py)
        last = self.take_last(a2)  # (B, d_model)
        last = self.head_drop(last)

        # 6. Classifier: d_model -> num_classes
        logits = self.fc(last)  # (B, num_classes)

        # Return logits or softmax
        if return_logits:
            return logits
        return self.softmax(logits)


Writing model/gclassifier_hybrid.py


In [12]:
%%writefile model/gclassifier_cnn_only.py
import torch
import torch.nn as nn

from .cnn_stem import CNNStem


class GalaxyClassifierCNNOnly(nn.Module):
    """
    CNN-only baseline: same CNNStem as GalaxyClassifierCNNS4D, but with NO
    S4D layers at all. Feature map is pooled directly (global average
    pooling) and classified.

    Why this model exists
    ----------------------
    Every result so far has tested "CNN stem -> S4D". Nothing so far has
    tested "CNN alone" at a matched parameter budget. That leaves an open
    question raised directly by the TA:

      1. Would a ~43K-param CNN alone reach similar accuracy to the
         ~55K-param CNN+S4D hybrid (86.80%)?
      2. Would a ~60K-param CNN alone reach similar accuracy to the
         ~63K-param CNN+S4D 3-layer hybrid (86.65%)?
      3. Is S4D actually adding signal beyond what the CNN stem alone
         already provides, or is the CNN stem doing all the work (echoing
         the color ablation finding, where the "obvious" explanation
         wasn't the real one)?
      4. How small can the CNN get before accuracy collapses -- i.e.
         where is the actual capacity floor for this task?

    This class, together with scripts/train_cnn_only.py, runs three sizes
    (~10K, ~43K, ~60K params) to answer all four questions with real
    numbers instead of assumptions on either side.

    Design choice: global average pooling over the stem's output grid is
    used as the CNN-only readout, because it is the direct analog of the
    S4D hybrid's mean-pooling over the Hilbert-scanned sequence -- same
    "average all spatial/sequence positions" idea, just without S4D's
    sequential processing in between. This keeps the comparison about
    S4D specifically, not about the readout strategy also changing.

    Parameters
    ----------
    num_classes : int, optional
        Number of output classes (default 4).
    colored : bool, optional
        RGB (3-channel) if True, grayscale (1-channel) if False. Default True.
    stem_reduction : int, optional
        Passed through to CNNStem, one of {4, 16}. Default 16 (matches the
        winning hybrid configuration).
    mid_channels : int, optional
        Stem hidden channel width. Default 32.
    d_model : int, optional
        Stem output channel width (and refine-conv width, if used). Default 64.
    stem_dropout : float, optional
        Dropout2d inside the stem. Default 0.1.
    head_dropout : float, optional
        Dropout applied to the pooled vector before the classifier head. Default 0.2.
    use_refine_conv : bool, optional
        If True, adds one extra 1x1 conv + GroupNorm + GELU after the stem,
        before pooling -- used to hit the ~43K/~60K parameter targets
        without changing the stem's own architecture. Default True.
    """

    def __init__(self, num_classes=4, colored=True, stem_reduction=16,
                 mid_channels=32, d_model=64, stem_dropout=0.1,
                 head_dropout=0.2, use_refine_conv=True):
        super().__init__()
        if stem_reduction not in (4, 16):
            raise ValueError(f"stem_reduction must be 4 or 16, got {stem_reduction}")

        self.hilbert_channels = 1 if not colored else 3
        self.d_model = d_model
        self.stem_reduction = stem_reduction
        self.use_refine_conv = use_refine_conv

        grid = 64 // (4 if stem_reduction == 16 else 2)
        self.seq_len = grid * grid  # kept for API/logging parity with the hybrid model

        self.cnn_stem = CNNStem(
            in_channels=self.hilbert_channels,
            d_model=d_model,
            mid_channels=mid_channels,
            reduction=stem_reduction,
            dropout=stem_dropout,
        )

        if use_refine_conv:
            # 1x1 conv: adds a small amount of extra depth/capacity so the
            # small/large variants can be tuned to specific parameter
            # budgets (~43K, ~60K) without changing the stem's own
            # architecture (which is the part already validated by the
            # earlier ablation).
            self.refine_conv = nn.Conv2d(d_model, d_model, kernel_size=1)
            groups = 8 if d_model % 8 == 0 else 1
            self.refine_norm = nn.GroupNorm(groups, d_model)
            self.refine_act = nn.GELU()
        else:
            self.refine_conv = None

        self.global_pool = nn.AdaptiveAvgPool2d(1)  # (B, d_model, H, W) -> (B, d_model, 1, 1)
        self.head_drop = nn.Dropout(head_dropout)
        self.fc = nn.Linear(d_model, num_classes)
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, x, return_logits=False):
        # x: (B, hilbert_channels, 64, 64)
        feat = self.cnn_stem(x)  # (B, d_model, grid, grid)

        if self.refine_conv is not None:
            feat = self.refine_act(self.refine_norm(self.refine_conv(feat)))

        pooled = self.global_pool(feat).flatten(1)  # (B, d_model)
        pooled = self.head_drop(pooled)

        logits = self.fc(pooled)  # (B, num_classes)

        if return_logits:
            return logits
        return self.softmax(logits)


Writing model/gclassifier_cnn_only.py


In [13]:
%%writefile model/functions.py
import os
import numpy as np

# PyTorch
import torch.nn.functional as F

# GalaxyMNIST dataset
from galaxy_mnist import GalaxyMNIST 

def load_data(root: str, download: bool = True, train: bool = True, colored: bool = False):
    """Load and preprocess GalaxyMNIST dataset.
    
    Parameters:
    -----------
    root : str
        Root directory where the dataset is stored or will be downloaded.
    download : bool
        Whether to download the dataset if not present.
    train : bool
        Whether to load the training set (True) or test set (False).
    colored : bool, optional
        Whether to use colored images (3 channels) or grayscale (1 channel).
        (default is False)
           
    Returns:
    --------
    X : torch.Tensor
        Preprocessed images of shape (N, 1, 64, 64) with pixel values in [0, 1] if grayscale,
        or (N, 3, 64, 64) if colored.
    y_onehot : torch.Tensor
        One-hot encoded labels of shape (N, num_classes).
    y : torch.Tensor
        Original labels of shape (N,).
    """
    dataset = GalaxyMNIST(root=root, download=download, train=train)
    print(f"Original Dataset Size: {len(dataset.data)} samples")

    # 1. Extract and process images: Mean across channels, Normalize to [0, 1]
    # Data shape is (N, 3, 64, 64) -> (N, 1, 64, 64)
    X = dataset.data.float()           # convert from uint8 -> float
    if not colored:
        X = X.mean(dim=1, keepdim=True)  # convert to grayscale by averaging channels
    X = X / 255.0                     # normalize to [0, 1]

    # 2. Extract targets and convert to one-hot encoding
    y = dataset.targets.long()
    # One hot encode the labels
    y_onehot = F.one_hot(y).float()
    return X, y_onehot, y

def format_row(values):
    """Formats a list of values, comma separated, even width"""
    return ", ".join(f"{v:10.6f}" if isinstance(v, (float, np.float32, np.float64)) else f"{v}" for v in values)

def export_model_parameters(model, output_dir="galaxy_s4_model_params"):
    """
    General function to export all model parameters AND buffers to .bin and .txt.
    Logic: (A, B, C) -> A blocks of B rows x C cols.
    Follows natural Row-Major storage order.
    """
    os.makedirs(output_dir, exist_ok=True)
    
    txt_path = os.path.join(output_dir, "weights.txt")
    bin_path = os.path.join(output_dir, "weights.bin")

    print(f"--- Exporting Model: {model.__class__.__name__} ---")

    state_dict = model.state_dict()

    with open(txt_path, "w") as f_txt, open(bin_path, "wb") as f_bin:
        for name, tensor in state_dict.items():
            shape = list(tensor.shape)
            print(f"Saving: {name:40} | Shape: {shape}")
            
            data_np = tensor.detach().cpu().contiguous().numpy()
            f_bin.write(data_np.astype(np.float32).tobytes())

            f_txt.write(f"[{name}] Shape: {shape}\n")

            if len(shape) == 0:
                f_txt.write(f"{data_np.item():10.6f}\n\n")

            elif len(shape) == 1:
                f_txt.write(format_row(data_np) + "\n\n")

            elif len(shape) == 2:
                rows, cols = shape
                for r in range(rows):
                    f_txt.write(format_row(data_np[r]) + "\n")
                f_txt.write("\n")

            elif len(shape) == 3:
                A, B, C = shape
                for a in range(A):
                    f_txt.write(f"# Block {a}\n")
                    for b in range(B):
                        row_values = data_np[a, b, :]
                        f_txt.write(format_row(row_values) + "\n")
                    f_txt.write("\n")
            
            elif len(shape) == 4:
                A, B, C, D = shape
                for a in range(A):
                    for b in range(B):
                        f_txt.write(f"# Block {a}, {b}\n")
                        for c in range(C):
                            f_txt.write(format_row(data_np[a, b, c, :]) + "\n")
                        f_txt.write("\n")

            else:
                f_txt.write(format_row(data_np.flatten()) + "\n\n")

    print(f"--- Export Complete. Files located in '{output_dir}' ---")

Writing model/functions.py


In [14]:
%%writefile model/interface.py
import torch

class ModelInterface:
    """
    Unified interface for galaxy classification models.
    
    This class abstracts the implementation details (Python vs RISC-V) and provides
    a consistent API for model inference regardless of backend.
    
    Parameters
    ----------
    implementation : str
        Either 'python' or 'riscv'.
    model_path : str
        Path to model weights (used for Python implementation).
    num_classes : int
        Number of output classes.
    colored : bool
        Whether model expects colored or grayscale images.
    device : torch.device
        Device for inference.
    
    Methods
    -------
    __call__(x)
        Run inference on input tensor x.
    """
    
    def __init__(self, implementation, model_path, num_classes, colored, device):
        """Initialize model based on implementation type."""
        self.implementation = implementation
        self.device = device
        
        if implementation == 'python':
            from model import GalaxyClassifierS4D
            print(f"Loading Python model from {model_path}")
            self.model = GalaxyClassifierS4D(colored=colored).to(device)
            self.model.load_state_dict(torch.load(model_path, map_location=device, weights_only=True))
            self.model.eval()
            
        elif implementation == 'riscv':
            print("Initializing RISC-V interface")
            # TODO: Setup RISC-V communication/configuration
            self.model = None
    
    def __call__(self, x):
        """
        Run model inference.
        
        Parameters
        ----------
        x : torch.Tensor
            Input tensor.
        
        Returns
        -------
        torch.Tensor
            Model predictions.
        """
        if self.implementation == 'python':
            return self.model(x)
        elif self.implementation == 'riscv':
            # TODO: Send x to RISC-V QEMU, receive predictions
            raise NotImplementedError("RISC-V inference not yet implemented")
    
    def eval(self):
        """Set model to evaluation mode (for consistency with PyTorch API)."""
        if self.implementation == 'python':
            self.model.eval()


Writing model/interface.py


In [15]:
%%writefile model/gui.py
"""
Interactive Galaxy Explorer GUI

This interactive visualization tool allows you to browse through the validation set and examine the 
model's predictions in real-time. The GUI displays each galaxy image using the Magma colormap 
(commonly used in astronomy visualization) alongside the model's softmax probability distribution
across all four classes.

Controls
--------
- LEFT/RIGHT Arrow Keys: Navigate through validation samples
- R Key: Jump to a random sample
- M Key: Toggle Magma colormap on/off
- Q Key: Quit the application

The visualization highlights the predicted class with a green bar, making it easy to spot correct 
classifications and identify failure cases where the model might confuse similar morphologies 
(e.g., smooth round vs. smooth cigar galaxies).
"""

import os
os.environ['PYGAME_HIDE_SUPPORT_PROMPT'] = '1'

import pygame
import numpy as np
import torch
import matplotlib.pyplot as plt

class GalaxyExplorerGUI:
    """
    Interactive GUI for exploring galaxy classifications.
    
    Displays galaxy images alongside model predictions with real-time navigation.
    Supports toggling between standard RGB and Magma colormap visualization.
    
    Parameters
    ----------
    model : ModelInterface
        Trained model for galaxy classification.
    x_val : torch.Tensor
        Validation images tensor of shape (N, C, H, W).
    y_val : torch.Tensor
        One-hot encoded validation labels of shape (N, num_classes).
    device : torch.device
        Device for running inference (CPU or CUDA).
    
    Attributes
    ----------
    current_idx : int
        Index of currently displayed sample.
    num_samples : int
        Total number of validation samples.
    predictions : np.ndarray
        Current model prediction probabilities.
    use_magma : bool
        Whether to apply Magma colormap to displayed image.
    """
    def __init__(self, model, x_val, y_val, device):
        self.model = model
        self.x_val = x_val  
        self.y_val = y_val  
        self.device = device
        
        self.current_idx = 0
        self.num_samples = len(x_val)
        self.predictions = np.zeros(4)
        
        # New toggle state
        self.use_magma = False
        
        pygame.init()
        self.WIDTH, self.HEIGHT = 1000, 600
        self.screen = pygame.display.set_mode((self.WIDTH, self.HEIGHT))
        pygame.display.set_caption("S4 GALAXY EXPLORER")
        
        self.CANVAS_SIZE = 448
        self.class_labels =  ["Smooth Round", "Smooth Cigar", "Edge-on Disk", "Unbarred Spiral"] # Class names for GalaxyMNIST
        
        self.COLOR_BG = (5, 5, 8)
        self.COLOR_ACCENT = (255, 160, 60) 
        self.COLOR_SUCCESS = (0, 255, 120)
        self.COLOR_FAILURE = (255, 50, 50)
        
        self.font = pygame.font.SysFont("monospace", 15)
        self.big_font = pygame.font.SysFont("monospace", 22, bold=True)
        
        self.cmap = plt.get_cmap('magma')
        self.update_sample(0)

    def update_sample(self, delta):
        """
        Update the currently displayed sample and compute predictions.
        
        Parameters
        ----------
        delta : int
            Offset to add to current index (wraps around at boundaries).
        """
        self.current_idx = (self.current_idx + delta) % self.num_samples
        self.model.eval()
        with torch.no_grad():
            img_tensor = self.x_val[self.current_idx].unsqueeze(0).to(self.device)
            probs = self.model(img_tensor)
            self.predictions = probs.squeeze().cpu().numpy()

    def draw(self):
        """
        Render the current frame of the GUI.
        
        Displays the galaxy image (with optional Magma colormap), prediction bars,
        sample metadata, and keyboard controls. Highlights correct predictions in
        green and incorrect predictions in red.
        """
        self.screen.fill(self.COLOR_BG)
        
        # x_val is always (N, C, H, W) -- so raw_img here is always channel-first,
        # never (H, W, C). The "or [H, W, C]" in the old comment was never actually
        # reachable, and assuming it was is what broke grayscale rendering below.
        raw_img = self.x_val[self.current_idx].numpy()
        
        if self.use_magma:
            # mean over the channel axis works whether C=1 (trivial mean, same as
            # squeezing) or C=3 (proper RGB->gray average) -- no need to special-case
            gray_img = raw_img.mean(axis=0)
            
            magma_img = self.cmap(gray_img) 
            rgb_render = (magma_img[:, :, :3] * 255).astype(np.uint8)
        else:
            # Standard RGB Render
            # Pygame wants (H, W, 3) uint8. Grayscale data is (1, H, W) -- transposing
            # alone gives (H, W, 1), which pygame's make_surface rejects (last dim must
            # be exactly 3), so we tile the single channel across R/G/B instead.
            rgb_render = raw_img.transpose(1, 2, 0)  # (C, H, W) -> (H, W, C)
            if rgb_render.shape[-1] == 1:
                rgb_render = np.repeat(rgb_render, 3, axis=-1)
            
            # Ensure uint8 [0, 255]
            if rgb_render.max() <= 1.0:
                rgb_render = (rgb_render * 255).astype(np.uint8)

        # 2. Create Pygame surface (Transpose from Row-Major to Width-Major)
        surface = pygame.surfarray.make_surface(rgb_render.transpose(1, 0, 2))
        scaled_img = pygame.transform.scale(surface, (self.CANVAS_SIZE, self.CANVAS_SIZE))
        self.screen.blit(scaled_img, (40, 60))

        pygame.draw.rect(self.screen, self.COLOR_ACCENT, (40, 60, self.CANVAS_SIZE, self.CANVAS_SIZE), 2)
        
        true_label_idx = torch.argmax(self.y_val[self.current_idx]).item()
        meta_txt = self.font.render(f"Sample: {self.current_idx} | Truth: {self.class_labels[true_label_idx]} | Magma: {'ON' if self.use_magma else 'OFF'}", True, (200, 200, 200))
        self.screen.blit(meta_txt, (40, 35))

        x_off = 520
        top_pred = np.argmax(self.predictions)
        self.screen.blit(self.big_font.render("MODEL PREDICTION", True, self.COLOR_ACCENT), (x_off, 60))
        
        for i, label in enumerate(self.class_labels):
            prob = self.predictions[i]
            bar_y = 120 + (i * 60)

            if i == top_pred and i == true_label_idx:
                color = self.COLOR_SUCCESS
            elif i == top_pred and i != true_label_idx:
                color = self.COLOR_FAILURE
            else:
                color = (150, 150, 150)

            txt = self.font.render(f"{label}: {prob*100:4.1f}%", True, color)
            self.screen.blit(txt, (x_off, bar_y))
            
            pygame.draw.rect(self.screen, (20, 20, 30), (x_off, bar_y + 25, 400, 15))
            pygame.draw.rect(self.screen, color, (x_off, bar_y + 25, int(prob * 400), 15))

        footer = self.font.render("[L/R] Change | [R] Rand | [M] Magma | [Q] Quit", True, self.COLOR_ACCENT)
        self.screen.blit(footer, (40, 530))

    def run(self):
        """
        Main event loop for the GUI application.
        
        Handles keyboard input for navigation (arrow keys, R for random),
        colormap toggling (M key), and quitting (Q key). Runs until the
        user closes the window or presses Q.
        """
        running = True
        while running:
            for event in pygame.event.get():
                if event.type == pygame.QUIT: running = False
                if event.type == pygame.KEYDOWN:
                    if event.key == pygame.K_RIGHT: self.update_sample(1)
                    if event.key == pygame.K_LEFT: self.update_sample(-1)
                    if event.key == pygame.K_r: self.update_sample(np.random.randint(0, self.num_samples))
                    if event.key == pygame.K_m: self.use_magma = not self.use_magma # Toggle
                    if event.key == pygame.K_q: running = False
            self.draw()
            pygame.display.flip()
        pygame.quit()



Writing model/gui.py


In [16]:
import inspect
from model.s4d_recurrent import S4D

src = inspect.getsource(S4D.forward)
print(src)
assert "torch.fft.rfft" in src and "torch.fft.irfft" in src, "Expected FFT convolution in S4D.forward"
print("CONFIRMED: S4D.forward (used by all four scripts) convolves via torch.fft.rfft/irfft —")
print("the same algorithm as S4DConv in notebook-best-s4d-model_1_.ipynb. No change was needed here.")


    def forward(self, u):
        """
        Forward pass through the S4D layer.
        
        Computes the convolution of the input sequence with the SSM kernel using FFT.
        The kernel is generated from the continuous-time SSM parameters and discretized
        using the learned timestep dt.
        
        Process:
        1. Materialize SSM parameters (dt, A, C) from log-space representations
        2. Generate discrete convolution kernel K via truncated power series
        3. Perform FFT-based convolution: y = K * u
        4. Add skip connection: y = y + D * u
        
        Parameters
        ----------
        u : torch.Tensor
            Input sequence of shape (B, H, L) if transposed=True, else (B, L, H).
            B : batch size
            H : feature dimension (d_model)
            L : sequence length
        
        Returns
        -------
        y : torch.Tensor
            Output sequence of same shape as input.
        None
            Placeholder for

/usr/local/lib/python3.12/dist-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


In [17]:
import torch
from model.tlts import TakeLastTimestep

layer = TakeLastTimestep()
x = torch.randn(3, 6, 2)
out = layer(x)
matches_last = torch.allclose(x[:, -1, :], out)
matches_mean = torch.allclose(x.mean(dim=1), out)
print(f"Output matches x[:, -1, :] (last timestep): {matches_last}")
print(f"Output matches x.mean(dim=1)  (mean pooling): {matches_mean}")
assert matches_last and not matches_mean, "TakeLastTimestep should take the last timestep, not the mean"
print("\nCONFIRMED: TakeLastTimestep now does genuine last-timestep pooling, matching its name/docstring")
print("and the reference notebook — not the mean-pooling it silently did in the uploaded zip.")


Output matches x[:, -1, :] (last timestep): True
Output matches x.mean(dim=1)  (mean pooling): False

CONFIRMED: TakeLastTimestep now does genuine last-timestep pooling, matching its name/docstring
and the reference notebook — not the mean-pooling it silently did in the uploaded zip.


In [18]:
# Sanity check: every model class these scripts use instantiates and forward-passes cleanly.
import torch
from model import GalaxyClassifierS4D, GalaxyClassifierCNNS4D, GalaxyClassifierCNNOnly

def count_params(m):
    return sum(p.numel() for p in m.parameters())

x_rgb = torch.randn(2, 3, 64, 64)
for name, m in [
    ("GalaxyClassifierS4D",     GalaxyClassifierS4D(colored=True)),
    ("GalaxyClassifierCNNS4D",  GalaxyClassifierCNNS4D(colored=True)),
    ("GalaxyClassifierCNNOnly", GalaxyClassifierCNNOnly(colored=True)),
]:
    out = m(x_rgb, return_logits=True)
    print(f"{name:24s} {count_params(m):>8,} params   out shape {tuple(out.shape)}")


GalaxyClassifierS4D        17,156 params   out shape (2, 4)
GalaxyClassifierCNNS4D     55,108 params   out shape (2, 4)
GalaxyClassifierCNNOnly    42,756 params   out shape (2, 4)


In [19]:
import math
import torch
import torch.nn as nn
from einops import repeat

class HilbertScan(nn.Module):
    """Reorders patches of a (B, C, H, W) image along a Hilbert curve."""
    def __init__(self, image_size=64, patch_size=1):
        super().__init__()
        assert image_size % patch_size == 0, "image_size must be divisible by patch_size"
        self.image_size = image_size
        self.patch_size = patch_size
        self.grid_size = image_size // patch_size
        self.num_patches = self.grid_size ** 2
        self.register_buffer("indices", self._get_hilbert_indices(self.grid_size))

    @staticmethod
    def _rot(s, x, y, rx, ry):
        if ry == 0:
            if rx == 1:
                x = s - 1 - x
                y = s - 1 - y
            x, y = y, x
        return x, y

    def _d2xy(self, n, d):
        x = y = 0
        t, s = d, 1
        while s < n:
            rx = (t // 2) & 1
            ry = (t ^ rx) & 1
            x, y = self._rot(s, x, y, rx, ry)
            x += s * rx
            y += s * ry
            t //= 4
            s *= 2
        return x, y

    def _get_hilbert_indices(self, grid_size):
        indices = []
        for d in range(grid_size * grid_size):
            x, y = self._d2xy(grid_size, d)
            indices.append(y * grid_size + x)
        return torch.LongTensor(indices)

    def forward(self, x):
        B, C, H, W = x.shape
        p = self.patch_size
        patches = x.unfold(2, p, p).unfold(3, p, p)
        patches = patches.permute(0, 2, 3, 1, 4, 5).contiguous()
        patches = patches.view(B, self.num_patches, C * p * p)
        return patches[:, self.indices, :]


class TakeLastTimestep(nn.Module):
    def forward(self, x):
        return x[:, -1, :]


class S4DConv(nn.Module):
    """Fast FFT-based parallel convolution S4D layer."""
    def __init__(self, d_model, d_state=64, dt_min=0.001, dt_max=0.1, transposed=True, lr=None):
        super().__init__()
        self.h = d_model
        self.n = d_state
        self.transposed = transposed

        log_dt = torch.rand(self.h) * (math.log(dt_max) - math.log(dt_min)) + math.log(dt_min)
        log_A_real = torch.log(0.5 * torch.ones(self.h, self.n // 2))
        A_imag = math.pi * repeat(torch.arange(self.n // 2), 'n -> h n', h=self.h)
        C_init = torch.randn(self.h, self.n // 2, dtype=torch.cfloat)

        self.register("log_dt", log_dt, lr)
        self.register("log_A_real", log_A_real, lr)
        self.register("A_imag", A_imag, lr)

        self.C = nn.Parameter(torch.view_as_real(C_init))
        self.D = nn.Parameter(torch.randn(self.h))

    def register(self, name, tensor, lr=None):
        if lr == 0.0:
            self.register_buffer(name, tensor)
        else:
            self.register_parameter(name, nn.Parameter(tensor))
            optim = {"weight_decay": 0.0}
            if lr is not None:
                optim["lr"] = lr
            setattr(getattr(self, name), "_optim", optim)

    def forward(self, u):
        if not self.transposed:
            u = u.transpose(-1, -2)
        L = u.size(-1)

        dt = torch.exp(self.log_dt)
        C = torch.view_as_complex(self.C)
        A = -torch.exp(self.log_A_real) + 1j * self.A_imag

        dtA = A * dt.unsqueeze(-1)
        K_exp = torch.exp(dtA.unsqueeze(-1) * torch.arange(L, device=u.device))
        C_tilde = C * (torch.exp(dtA) - 1.) / A
        k = 2 * torch.einsum('hn, hnl -> hl', C_tilde, K_exp).real

        k_f = torch.fft.rfft(k, n=2 * L)
        u_f = torch.fft.rfft(u, n=2 * L)
        y = torch.fft.irfft(u_f * k_f, n=2 * L)[..., :L]
        y = y + u * self.D.unsqueeze(-1)

        if not self.transposed:
            y = y.transpose(-1, -2)
        return y, None


class ConvPatchStem(nn.Module):
    """Convolutional stem for local neighborhood mixing before patch projection."""
    def __init__(self, in_channels, d_model, patch_size):
        super().__init__()
        mid_channels = max(in_channels * 8, 32)
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, kernel_size=3, stride=1, padding=1),
            nn.GELU(),
            nn.Conv2d(mid_channels, d_model, kernel_size=patch_size, stride=patch_size),
        )

    def forward(self, x):
        return self.net(x)


class MainStudyGalaxyClassifier(nn.Module):
    """Production S4D Classifier."""
    def __init__(self, s4_state=64, d_model=64, num_classes=4, colored=True,
                 num_layers=2, patch_size=1, pooling="last",
                 use_norm=False, use_residual=False, dropout=0.0,
                 patch_embed="linear"):
        super().__init__()
        self.hilbert_channels = 1 if not colored else 3
        self.patch_size = patch_size
        self.pooling = pooling
        self.use_norm = use_norm
        self.use_residual = use_residual
        self.patch_embed = patch_embed

        if patch_embed == "linear":
            self.hilbert_scan = HilbertScan(image_size=64, patch_size=patch_size)
            patch_dim = self.hilbert_channels * patch_size * patch_size
            self.uproject = nn.Linear(patch_dim, d_model)
            self.conv_stem = None
        elif patch_embed == "conv":
            self.conv_stem = ConvPatchStem(self.hilbert_channels, d_model, patch_size)
            self.hilbert_scan = HilbertScan(image_size=64 // patch_size, patch_size=1)
            self.uproject = nn.Identity()

        self.s4_layers = nn.ModuleList([
            S4DConv(d_model=d_model, d_state=s4_state, transposed=False)
            for _ in range(num_layers)
        ])
        self.acts = nn.ModuleList([nn.GELU() for _ in range(num_layers)])
        self.norms = nn.ModuleList([nn.LayerNorm(d_model) for _ in range(num_layers)]) if use_norm else None
        self.drop = nn.Dropout(dropout) if dropout > 0 else nn.Identity()

        if pooling == "last":
            self.take_last = TakeLastTimestep()
        elif pooling == "mean":
            self.take_last = None
        else:
            raise ValueError(f"Unknown pooling type {pooling}")

        self.fc = nn.Linear(d_model, num_classes)
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, x, return_logits=True):
        if self.patch_embed == "conv":
            feat = self.conv_stem(x)
            x_seq = self.hilbert_scan(feat)
            h = self.uproject(x_seq)
        else:
            x_seq = self.hilbert_scan(x)
            h = self.uproject(x_seq)

        for i, (s4_layer, act) in enumerate(zip(self.s4_layers, self.acts)):
            residual = h
            h_in = self.norms[i](h) if self.use_norm else h
            h_out, _ = s4_layer(h_in)
            h_out = act(h_out)
            h_out = self.drop(h_out)
            h = residual + h_out if self.use_residual else h_out

        pooled = h.mean(dim=1) if self.take_last is None else self.take_last(h)
        logits = self.fc(pooled)

        if return_logits:
            return logits
        return self.softmax(logits)

# S4D Future-Work Validation — Kaggle Experiment Suite

This notebook operationalizes the four unverified follow-ups identified in the report:

1. Re-run the richer-stem family's full **4×3 stem-depth × S4D-layer grid** under the **main 630-epoch recipe**.
2. Add repeat seeds to the production family's **full-scale and closest-budget conv-vs-linear comparisons**, and to both **pooling comparisons**.
3. Repeat the richer-stem family's **Gaussian input-noise robustness sweep** with an independently trained seed.
4. Test whether the richer-stem `GroupNorm` accuracy survives a **fixed-statistics inference fold into the preceding convolutions**, as a portability-oriented approximation. True `GroupNorm` is input-dependent and therefore cannot be folded exactly like BatchNorm; this notebook measures the accuracy cost of replacing it with calibrated fixed group statistics, then folds the resulting affine transforms into the convolution weights.

All runs use the report's **main recipe** unless a cell explicitly says otherwise:
**AdamW, lr=1e-3, 1e-2 decay on ordinary weights, no decay on bias/norm/S4D-special parameters, cosine warm restarts, T0=10, Tmult=2, eta_min=1e-5, gradient clipping at 1.0, batch size 32, 630 epochs, best-validation checkpoint used for test evaluation.**

The notebook is cacheable and resumable: completed run JSON/weights are reused rather than silently retrained.

In [20]:
# ---------------------------
# Experiment switches
# ---------------------------
RUN_RICHER_GRID_MAIN = True
RUN_PRODUCTION_REPEATS = True
RUN_NOISE_REPEAT = True
RUN_GN_FOLD = True
RUN_STRICT_NEAR_MATCH_BUDGET_PAIR = False  # optional d=72 conv vs d=103 linear

# Original report seed + one independent repeat used throughout this notebook.
PRODUCTION_SEEDS = [30485, 8842]
RICHER_GRID_MAIN_SEED = 30485
NOISE_REPEAT_SEED = 8842
GN_FOLD_SEED = 30485

# Smoke-test mode is for pipeline validation only.
SMOKE_TEST = False
SMOKE_EPOCHS = 3

# Noise sweep. The report used these sigmas; clipping is configurable because the report
# text does not state whether the original implementation clipped noisy pixels back to [0,1].
NOISE_SIGMAS = [0.0, 0.05, 0.10, 0.20, 0.30]
NOISE_CLIP_TO_UNIT = True

# Noise robustness uses the richer grid's 13 trained models, including the raw-pixel S4D-only baseline.
NOISE_EVAL_BATCH_SIZE = 64

# Reuse completed outputs. Set False only when intentionally rebuilding the suite.
USE_CACHE = True

BASE_DIR = Path('/kaggle/working/s4d_future_work')
RESULTS_DIR = BASE_DIR / 'results'
WEIGHTS_DIR = BASE_DIR / 'weights'
PLOTS_DIR = BASE_DIR / 'plots'
EXPORT_DIR = BASE_DIR / 'exports'
for d in [RESULTS_DIR, WEIGHTS_DIR, PLOTS_DIR, EXPORT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('Output root:', BASE_DIR)
print('CUDA:', torch.cuda.is_available(), '| device:', DEVICE if 'DEVICE' in globals() else 'initialising')

Output root: /kaggle/working/s4d_future_work
CUDA: True | device: initialising


In [21]:
# ---------------------------
# Reproducibility + data
# ---------------------------
import os, json, math, time, random, copy, re, shutil, hashlib
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score, confusion_matrix

from model.functions import load_data

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
CLASS_NAMES = ['Smooth Round', 'Smooth Cigar', 'Edge-on Disk', 'Unbarred Spiral']
SPLIT_SEED = 30485
MAIN_EPOCHS = 630


def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    # Determinism is intentionally not forced: the report's repeat-seed noise estimate is
    # meant to include ordinary run-to-run variation on the Kaggle GPU/software stack.


def make_augmented_dataset(X, y):
    class AugmentedGalaxyDataset(Dataset):
        def __init__(self, X, y):
            self.X, self.y = X, y
        def __len__(self):
            return len(self.X)
        def __getitem__(self, idx):
            img = self.X[idx]
            label = self.y[idx]
            k = random.randint(0, 3)
            if k:
                img = torch.rot90(img, k, dims=(1, 2))
            if random.random() < 0.5:
                img = torch.flip(img, dims=(2,))
            if random.random() < 0.5:
                img = torch.flip(img, dims=(1,))
            return img, label
    return AugmentedGalaxyDataset(X, y)

X, y_onehot, y = load_data(root='./data', download=True, train=True, colored=True)
X_test, y_test_onehot, y_test = load_data(root='./data', download=True, train=False, colored=True)
x_train, x_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=SPLIT_SEED, stratify=y
)
DATA_SPLIT = {
    'train': (x_train, y_train),
    'val': (x_val, y_val),
    'test': (X_test, y_test),
}
print('RGB train/val/test:', x_train.shape, x_val.shape, X_test.shape)
print('Shared split seed:', SPLIT_SEED)

  0%|          | 0.00/68.7M [00:00<?, ?B/s]

  0%|          | 32.8k/68.7M [00:00<04:45, 241kB/s]

  0%|          | 65.5k/68.7M [00:00<04:48, 238kB/s]

  0%|          | 164k/68.7M [00:00<02:31, 454kB/s] 

  1%|          | 360k/68.7M [00:00<01:21, 835kB/s]

...

 88%|████████▊ | 15.2M/17.3M [00:02<00:00, 13.1MB/s]

100%|██████████| 17.3M/17.3M [00:02<00:00, 8.43MB/s]

Original Dataset Size: 8000 samples


Original Dataset Size: 2000 samples


RGB train/val/test: torch.Size([6400, 3, 64, 64]) torch.Size([1600, 3, 64, 64]) torch.Size([2000, 3, 64, 64])
Shared split seed: 30485


## Experiment 1 — richer-stem full grid under the main recipe

The report's richer-stem grid contains 13 models: stem depth 1–4 crossed with 0/1/2 S4D layers, plus the raw-pixel S4D-only baseline. The reconstruction below is checked against the report's published parameter counts and sequence lengths before any training begins.

In [22]:
# ---------------------------
# Exact-by-report richer-stem reconstruction
# ---------------------------
class RicherStem(nn.Module):
    """Reconstruction of the report's 1/2/3/4-layer richer stems.

    The structural variants are chosen to reproduce the report's published parameter counts:
      depth 1 -> 2,180 params incl. classifier
      depth 2 -> 19,844
      depth 3 -> 29,156
      depth 4 -> 38,468

    All convolution outputs are GroupNorm + GELU. The 4-layer variant includes the documented
    full-resolution residual block. Spatial side length is 64 for depth 1 and 16 for depths 2-4.
    """
    def __init__(self, depth, in_channels=3, d_model=64, mid_channels=32, dropout=0.1):
        super().__init__()
        if depth not in (1,2,3,4):
            raise ValueError(depth)
        self.depth = depth
        self.out_channels = d_model
        self.grid = 64 if depth == 1 else 16
        self.dropout = nn.Identity()

        def gn(ch):
            groups = 8 if ch % 8 == 0 else 1
            return nn.GroupNorm(groups, ch)

        if depth == 1:
            self.conv1 = nn.Conv2d(in_channels, d_model, 3, stride=1, padding=1)
            self.norm1 = gn(d_model)
            self.act1 = nn.GELU()
        elif depth == 2:
            self.conv1 = nn.Conv2d(in_channels, mid_channels, 3, stride=1, padding=1)
            self.norm1 = gn(mid_channels)
            self.act1 = nn.GELU()
            self.conv2 = nn.Conv2d(mid_channels, d_model, 3, stride=4, padding=1)
            self.norm2 = gn(d_model)
            self.act2 = nn.GELU()
        elif depth == 3:
            self.conv1 = nn.Conv2d(in_channels, mid_channels, 3, stride=1, padding=1)
            self.norm1 = gn(mid_channels)
            self.act1 = nn.GELU()
            self.conv2 = nn.Conv2d(mid_channels, mid_channels, 3, stride=2, padding=1)
            self.norm2 = gn(mid_channels)
            self.act2 = nn.GELU()
            self.conv3 = nn.Conv2d(mid_channels, d_model, 3, stride=2, padding=1)
            self.norm3 = gn(d_model)
            self.act3 = nn.GELU()
        else:
            # This matches the research CNNStem already used in the uploaded notebook:
            # full-res detail conv + full-res residual conv + two stride-2 downsamples.
            self.conv1 = nn.Conv2d(in_channels, mid_channels, 3, stride=1, padding=1)
            self.norm1 = gn(mid_channels)
            self.act1 = nn.GELU()
            self.res_conv = nn.Conv2d(mid_channels, mid_channels, 3, stride=1, padding=1)
            self.res_norm = gn(mid_channels)
            self.res_act = nn.GELU()
            self.drop = nn.Dropout2d(dropout)
            self.conv2 = nn.Conv2d(mid_channels, mid_channels, 3, stride=2, padding=1)
            self.norm2 = gn(mid_channels)
            self.act2 = nn.GELU()
            self.conv3 = nn.Conv2d(mid_channels, d_model, 3, stride=2, padding=1)
            self.norm3 = gn(d_model)
            self.act3 = nn.GELU()

    def forward(self, x):
        if self.depth == 1:
            return self.act1(self.norm1(self.conv1(x)))
        if self.depth == 2:
            x = self.act1(self.norm1(self.conv1(x)))
            return self.act2(self.norm2(self.conv2(x)))
        if self.depth == 3:
            x = self.act1(self.norm1(self.conv1(x)))
            x = self.act2(self.norm2(self.conv2(x)))
            return self.act3(self.norm3(self.conv3(x)))
        x = self.act1(self.norm1(self.conv1(x)))
        r = self.res_act(self.res_norm(self.res_conv(x)))
        x = self.drop(x + r)
        x = self.act2(self.norm2(self.conv2(x)))
        return self.act3(self.norm3(self.conv3(x)))


class RicherGridModel(nn.Module):
    def __init__(self, stem_depth, num_s4_layers, d_model=64, s4_state=64, num_classes=4, stem_dropout=0.1):
        super().__init__()
        if num_s4_layers not in (0,1,2):
            raise ValueError(num_s4_layers)
        self.stem_depth = stem_depth
        self.num_s4_layers = num_s4_layers
        self.cnn_stem = RicherStem(stem_depth, d_model=d_model, dropout=stem_dropout)
        self.grid = self.cnn_stem.grid
        self.seq_len = self.grid * self.grid
        from model.hilbert import HilbertScan as PackageHilbertScan
        self.hilbert_scan = PackageHilbertScan(n=self.grid)
        self.s4_layers = nn.ModuleList([
            S4D(d_model=d_model, d_state=s4_state, transposed=False)
            for _ in range(num_s4_layers)
        ])
        self.acts = nn.ModuleList([nn.GELU() for _ in range(num_s4_layers)])
        self.take_last = TakeLastTimestep()
        self.fc = nn.Linear(d_model, num_classes)

    def forward(self, x, return_logits=True):
        feat = self.cnn_stem(x)
        h = self.hilbert_scan(feat)
        for layer, act in zip(self.s4_layers, self.acts):
            h, _ = layer(h)
            h = act(h)
        pooled = self.take_last(h)
        logits = self.fc(pooled)
        return logits if return_logits else torch.softmax(logits, dim=-1)


# Safer explicit constructor using the package HilbertScan(n=...) from the uploaded notebook.
def make_richer_grid_model(stem_depth, num_s4_layers):
    return RicherGridModel(stem_depth, num_s4_layers)


RICHER_REPORT_PARAMS = {
    (1,0): 2180, (1,1): 10500, (1,2): 18820,
    (2,0): 19844, (2,1): 28164, (2,2): 36484,
    (3,0): 29156, (3,1): 37476, (3,2): 45796,
    (4,0): 38468, (4,1): 46788, (4,2): 55108,
}
RICHER_REPORT_SEQ = {1: 4096, 2: 256, 3: 256, 4: 256}

checks=[]
for depth in (1,2,3,4):
    for s4_layers in (0,1,2):
        m = make_richer_grid_model(depth, s4_layers)
        params = sum(p.numel() for p in m.parameters())
        checks.append({
            'stem_depth': depth, 's4_layers': s4_layers,
            'actual_params': params,
            'report_params': RICHER_REPORT_PARAMS[(depth,s4_layers)],
            'actual_seq_len': m.seq_len,
            'report_seq_len': RICHER_REPORT_SEQ[depth],
            'ok': params == RICHER_REPORT_PARAMS[(depth,s4_layers)] and m.seq_len == RICHER_REPORT_SEQ[depth],
        })
check_df = pd.DataFrame(checks)
display(check_df)
if not check_df['ok'].all():
    raise RuntimeError('Richer-grid reconstruction did not reproduce the report parameter/sequence checks.')

# Raw-pixel baseline from the report: 17,156 params, seq_len=4096, 2 S4D layers.
raw_baseline = GalaxyClassifierS4D(s4_state=64, d_model=64, num_classes=4, colored=True)
assert sum(p.numel() for p in raw_baseline.parameters()) == 17156
print('Raw-pixel S4D baseline: params=17,156, seq_len=4096')

,stem_depth,s4_layers,actual_params,report_params,actual_seq_len,report_seq_len,ok
0,1,0,2180,2180,4096,4096,True
1,1,1,10500,10500,4096,4096,True
2,1,2,18820,18820,4096,4096,True
3,2,0,19844,19844,256,256,True
4,2,1,28164,28164,256,256,True
5,2,2,36484,36484,256,256,True
6,3,0,29156,29156,256,256,True
7,3,1,37476,37476,256,256,True
8,3,2,45796,45796,256,256,True
9,4,0,38468,38468,256,256,True


Raw-pixel S4D baseline: params=17,156, seq_len=4096


In [23]:
# ---------------------------
# Main-recipe training/evaluation engine
# ---------------------------
MAIN_RECIPE = {
    'batch_size': 32,
    'lr': 1e-3,
    'weight_decay': 1e-2,
    'epochs': MAIN_EPOCHS,
    'label_smoothing': 0.0,
    'grad_clip': 1.0,
}


def make_main_optimizer(model):
    decay_params, no_decay_params, special_params = [], [], []
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if hasattr(param, '_optim'):
            special_params.append({
                'params': [param],
                'lr': getattr(param, '_optim', {}).get('lr', MAIN_RECIPE['lr']),
                'weight_decay': 0.0,
            })
        elif any(k in name.lower() for k in ['bias', 'norm', 'layernorm']):
            no_decay_params.append(param)
        else:
            decay_params.append(param)
    return torch.optim.AdamW(
        [
            {'params': decay_params, 'lr': MAIN_RECIPE['lr'], 'weight_decay': MAIN_RECIPE['weight_decay']},
            {'params': no_decay_params, 'lr': MAIN_RECIPE['lr'], 'weight_decay': 0.0},
        ] + special_params
    )


def make_main_scheduler(opt):
    return torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        opt, T_0=10, T_mult=2, eta_min=1e-5
    )


def macro_metrics(y_true, y_pred, probs):
    return {
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'f1_macro': float(f1_score(y_true, y_pred, average='macro')),
        'precision_macro': float(precision_score(y_true, y_pred, average='macro', zero_division=0)),
        'recall_macro': float(recall_score(y_true, y_pred, average='macro', zero_division=0)),
        'roc_auc_macro': float(roc_auc_score(y_true, probs, multi_class='ovr', average='macro')),
    }


def evaluate_model(model, loader):
    model.eval()
    all_targets, all_preds, all_probs = [], [], []
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            logits = model(images, return_logits=True)
            probs = torch.softmax(logits, dim=1)
            preds = torch.argmax(logits, dim=1)
            all_targets.extend(labels.cpu().numpy().tolist())
            all_preds.extend(preds.cpu().numpy().tolist())
            all_probs.extend(probs.cpu().numpy().tolist())
    m = macro_metrics(np.array(all_targets), np.array(all_preds), np.array(all_probs))
    m['confusion_matrix'] = confusion_matrix(all_targets, all_preds).tolist()
    return m


def run_id(slug, seed):
    return f'{slug}__main__seed{seed}'


def build_loaders(seed):
    set_all_seeds(seed)
    train_x, train_y = DATA_SPLIT['train']
    val_x, val_y = DATA_SPLIT['val']
    test_x, test_y = DATA_SPLIT['test']
    train_ds = make_augmented_dataset(train_x, train_y)
    return (
        DataLoader(train_ds, batch_size=MAIN_RECIPE['batch_size'], shuffle=True),
        DataLoader(TensorDataset(val_x, val_y), batch_size=MAIN_RECIPE['batch_size'], shuffle=False),
        DataLoader(TensorDataset(test_x, test_y), batch_size=64, shuffle=False),
    )


def save_curve(history, out_path, title):
    fig, ax = plt.subplots(figsize=(7,4))
    ax.plot(history['train_acc'], label='Train Acc')
    ax.plot(history['val_acc'], label='Val Acc')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy'); ax.set_title(title); ax.legend()
    fig.tight_layout(); fig.savefig(out_path, dpi=150); plt.close(fig)


def train_model(spec, seed, purpose='main'):
    slug = spec['id']
    rid = run_id(slug, seed)
    result_path = RESULTS_DIR / f'{rid}.json'
    weights_path = WEIGHTS_DIR / f'{rid}.pt'
    curve_path = PLOTS_DIR / f'{rid}.png'
    if USE_CACHE and result_path.exists() and weights_path.exists():
        result = json.loads(result_path.read_text())
        result['_cached'] = True
        return result

    set_all_seeds(seed)
    train_loader, val_loader, test_loader = build_loaders(seed)
    model = spec['builder']().to(DEVICE)
    expected = spec.get('expected_params')
    actual_params = sum(p.numel() for p in model.parameters())
    if expected is not None and actual_params != expected:
        raise RuntimeError(f'{slug}: expected {expected} params, got {actual_params}')

    opt = make_main_optimizer(model)
    sched = make_main_scheduler(opt)
    loss_fn = nn.CrossEntropyLoss(label_smoothing=0.0)
    epochs = SMOKE_EPOCHS if SMOKE_TEST else MAIN_EPOCHS
    history = {'train_loss': [], 'train_acc': [], 'val_acc': [], 'lr': []}
    best_val = -1.0
    best_state = None
    t0 = time.time()

    for epoch in range(epochs):
        model.train()
        total = correct = 0
        running_loss = 0.0
        for images, labels in train_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            logits = model(images, return_logits=True)
            loss = loss_fn(logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAIN_RECIPE['grad_clip'])
            opt.step()
            running_loss += float(loss.item()) * labels.size(0)
            correct += int((torch.argmax(logits, dim=1) == labels).sum())
            total += labels.size(0)
        sched.step()
        val = evaluate_model(model, val_loader)
        history['train_loss'].append(running_loss / max(1,total))
        history['train_acc'].append(correct / max(1,total))
        history['val_acc'].append(val['accuracy'])
        history['lr'].append(float(opt.param_groups[0]['lr']))
        if val['accuracy'] > best_val:
            best_val = val['accuracy']
            best_state = copy.deepcopy(model.state_dict())
        if SMOKE_TEST or (epoch + 1) % 25 == 0 or epoch == epochs-1:
            print(f'[{rid}] epoch {epoch+1}/{epochs} train={history["train_acc"][-1]:.4f} val={val["accuracy"]:.4f}')

    model.load_state_dict(best_state)
    test = evaluate_model(model, test_loader)
    elapsed = time.time() - t0
    torch.save(model.state_dict(), weights_path)
    save_curve(history, curve_path, spec['label'])
    result = {
        'run_id': rid,
        'purpose': purpose,
        'architecture': slug,
        'label': spec['label'],
        'seed': seed,
        'params': actual_params,
        'expected_params': expected,
        'epochs': epochs,
        'train_time_sec': elapsed,
        'accuracy': test['accuracy'],
        'f1_macro': test['f1_macro'],
        'precision_macro': test['precision_macro'],
        'recall_macro': test['recall_macro'],
        'roc_auc_macro': test['roc_auc_macro'],
        'confusion_matrix': test['confusion_matrix'],
        'history': history,
        'weights_file': str(weights_path),
        'recipe': 'main',
        'split_seed': SPLIT_SEED,
        'purpose_detail': purpose,
    }
    result_path.write_text(json.dumps(result, indent=2))
    return result


def result_frame(results):
    return pd.DataFrame([{k:r.get(k) for k in ['run_id','architecture','label','seed','params','accuracy','f1_macro','precision_macro','recall_macro','roc_auc_macro','train_time_sec']} for r in results])

### Production-family comparison registry

The production-family comparisons below reproduce the report's settings:

- **Full scale:** `d_model=256`, 3 S4D layers, conv vs linear, last pooling.
- **Pooling closure:** the same full-scale conv and linear models, each with last vs mean pooling.
- **Closest budget pair used in the report:** conv `d=72` (69,660 params) vs linear `d=108` (76,360 params). The report calls this the closest available pair rather than perfectly parameter-matched.
- An optional **strict near-match** is also provided: linear `d=103` has 69,117 params, within 543 parameters of the conv `d=72` model.

In [24]:
# ---------------------------
# Production-family models
# ---------------------------
# MainStudyGalaxyClassifier is defined in the earlier compatibility cell copied from the uploaded notebook.

def prod_builder(patch_embed, d_model, num_layers=3, pooling='last'):
    return lambda: MainStudyGalaxyClassifier(
        s4_state=d_model,
        d_model=d_model,
        num_classes=4,
        colored=True,
        num_layers=num_layers,
        patch_size=4,
        pooling=pooling,
        use_norm=False,
        use_residual=False,
        dropout=0.0,
        patch_embed=patch_embed,
    )

PROD_SPECS = [
    {'id':'prod_conv_d256_last', 'label':'Production Conv d256, 3L, last', 'builder':prod_builder('conv',256,3,'last'), 'expected_params':528004, 'group':'fullscale'},
    {'id':'prod_linear_d256_last', 'label':'Production Linear d256, 3L, last', 'builder':prod_builder('linear',256,3,'last'), 'expected_params':408324, 'group':'fullscale'},
    {'id':'prod_conv_d256_mean', 'label':'Production Conv d256, 3L, mean', 'builder':prod_builder('conv',256,3,'mean'), 'expected_params':528004, 'group':'pooling'},
    {'id':'prod_linear_d256_mean', 'label':'Production Linear d256, 3L, mean', 'builder':prod_builder('linear',256,3,'mean'), 'expected_params':408324, 'group':'pooling'},
    {'id':'prod_conv_d72_last', 'label':'Production Conv d72, 3L, last (~69.7K)', 'builder':prod_builder('conv',72,3,'last'), 'expected_params':69660, 'group':'budget'},
    {'id':'prod_linear_d108_last', 'label':'Production Linear d108, 3L, last (~76.4K)', 'builder':prod_builder('linear',108,3,'last'), 'expected_params':76360, 'group':'budget'},
]
if RUN_STRICT_NEAR_MATCH_BUDGET_PAIR:
    PROD_SPECS.append({'id':'prod_linear_d103_last', 'label':'Production Linear d103, 3L, last (~69.1K)', 'builder':prod_builder('linear',103,3,'last'), 'expected_params':69117, 'group':'strict_budget'})

prod_checks=[]
for s in PROD_SPECS:
    m=s['builder'](); n=sum(p.numel() for p in m.parameters())
    prod_checks.append({'id':s['id'],'actual':n,'expected':s['expected_params'],'ok':n==s['expected_params']})
display(pd.DataFrame(prod_checks))
if not all(x['ok'] for x in prod_checks):
    raise RuntimeError('Production model parameter check failed.')

,id,actual,expected,ok
0,prod_conv_d256_last,528004,528004,True
1,prod_linear_d256_last,408324,408324,True
2,prod_conv_d256_mean,528004,528004,True
3,prod_linear_d256_mean,408324,408324,True
4,prod_conv_d72_last,69660,69660,True
5,prod_linear_d108_last,76360,76360,True


In [25]:
# ---------------------------
# Run production repeat-seed comparisons
# ---------------------------
production_results=[]
if RUN_PRODUCTION_REPEATS:
    for spec in PROD_SPECS:
        for seed in PRODUCTION_SEEDS:
            production_results.append(train_model(spec, seed, purpose='production_repeat_seed'))
    production_df = result_frame(production_results)
    display(production_df.sort_values(['architecture','seed']))
else:
    production_df = pd.DataFrame()
    print('Production repeat suite skipped.')

[prod_conv_d256_last__main__seed30485] epoch 25/630 train=0.8013 val=0.7869


[prod_conv_d256_last__main__seed30485] epoch 50/630 train=0.8163 val=0.7937


[prod_conv_d256_last__main__seed30485] epoch 75/630 train=0.8164 val=0.7831


[prod_conv_d256_last__main__seed30485] epoch 100/630 train=0.8438 val=0.7987


[prod_conv_d256_last__main__seed30485] epoch 125/630 train=0.8764 val=0.8169


...

[prod_linear_d108_last__main__seed8842] epoch 575/630 train=0.9047 val=0.8131


[prod_linear_d108_last__main__seed8842] epoch 600/630 train=0.9033 val=0.8150


[prod_linear_d108_last__main__seed8842] epoch 625/630 train=0.9031 val=0.8144


[prod_linear_d108_last__main__seed8842] epoch 630/630 train=0.9050 val=0.8137


,run_id,architecture,label,seed,params,accuracy,f1_macro,precision_macro,recall_macro,roc_auc_macro,train_time_sec
1,prod_conv_d256_last__main__seed8842,prod_conv_d256_last,"Production Conv d256, 3L, last",8842,528004,0.8380,0.838081,0.838324,0.838557,0.965685,4138.312827
0,prod_conv_d256_last__main__seed30485,prod_conv_d256_last,"Production Conv d256, 3L, last",30485,528004,0.8355,0.836426,0.836814,0.836082,0.966084,4133.046779
5,prod_conv_d256_mean__main__seed8842,prod_conv_d256_mean,"Production Conv d256, 3L, mean",8842,528004,0.8470,0.847628,0.847855,0.847519,0.969154,4162.025631
4,prod_conv_d256_mean__main__seed30485,prod_conv_d256_mean,"Production Conv d256, 3L, mean",30485,528004,0.8420,0.842361,0.843541,0.842249,0.969804,4160.711899
9,prod_conv_d72_last__main__seed8842,prod_conv_d72_last,"Production Conv d72, 3L, last (~69.7K)",8842,69660,0.8355,0.835994,0.835811,0.836191,0.965781,1836.374969
8,prod_conv_d72_last__main__seed30485,prod_conv_d72_last,"Production Conv d72, 3L, last (~69.7K)",30485,69660,0.8360,0.836480,0.837302,0.836144,0.965499,1835.528237
11,prod_linear_d108_last__main__seed8842,prod_linear_d108_last,"Production Linear d108, 3L, last (~76.4K)",8842,76360,0.8100,0.810213,0.811283,0.810638,0.960243,1768.996667
10,prod_linear_d108_last__main__seed30485,prod_linear_d108_last,"Production Linear d108, 3L, last (~76.4K)",30485,76360,0.7965,0.796642,0.798438,0.796428,0.954953,1770.566208
3,prod_linear_d256_last__main__seed8842,prod_linear_d256_last,"Production Linear d256, 3L, last",8842,408324,0.8210,0.821656,0.822023,0.821414,0.963172,3689.455492
2,prod_linear_d256_last__main__seed30485,prod_linear_d256_last,"Production Linear d256, 3L, last",30485,408324,0.8170,0.817840,0.818223,0.817675,0.959543,3682.499867


In [26]:
# ---------------------------
# Paired summaries: conv-vs-linear and pooling
# ---------------------------
def paired_summary(df, a_id, b_id, name, metric='accuracy'):
    a=df[df.architecture==a_id][['seed',metric]].rename(columns={metric:'a'})
    b=df[df.architecture==b_id][['seed',metric]].rename(columns={metric:'b'})
    m=a.merge(b,on='seed',how='inner')
    if len(m)==0:
        return None
    m['gap_pp']=(m['a']-m['b'])*100
    return {
        'comparison':name,
        'n_seeds':len(m),
        'mean_a':m.a.mean(), 'mean_b':m.b.mean(),
        'mean_gap_pp':m.gap_pp.mean(),
        'std_gap_pp':m.gap_pp.std(ddof=1) if len(m)>1 else np.nan,
        'min_gap_pp':m.gap_pp.min(), 'max_gap_pp':m.gap_pp.max(),
        'same_direction_count':int((m.gap_pp>0).sum()),
        'seed_rows':m.to_dict('records'),
    }

comparisons=[]
if len(production_df):
    for a,b,name in [
        ('prod_conv_d256_last','prod_linear_d256_last','Full-scale conv - linear (last pool)'),
        ('prod_conv_d256_mean','prod_linear_d256_mean','Full-scale conv - linear (mean pool)'),
        ('prod_conv_d72_last','prod_linear_d108_last','Closest-budget conv - linear (report pair)'),
    ]:
        x=paired_summary(production_df,a,b,name)
        if x: comparisons.append(x)
    if RUN_STRICT_NEAR_MATCH_BUDGET_PAIR:
        x=paired_summary(production_df,'prod_conv_d72_last','prod_linear_d103_last','Near-exact budget conv - linear')
        if x: comparisons.append(x)
    for a,b,name in [
        ('prod_conv_d256_last','prod_conv_d256_mean','Conv d256: last - mean pooling'),
        ('prod_linear_d256_last','prod_linear_d256_mean','Linear d256: last - mean pooling'),
    ]:
        x=paired_summary(production_df,a,b,name)
        if x: comparisons.append(x)

comparison_df=pd.DataFrame([{k:v for k,v in x.items() if k!='seed_rows'} for x in comparisons])
display(comparison_df)
for x in comparisons:
    print('\n',x['comparison'])
    display(pd.DataFrame(x['seed_rows']))

,comparison,n_seeds,mean_a,mean_b,mean_gap_pp,std_gap_pp,min_gap_pp,max_gap_pp,same_direction_count
0,Full-scale conv - linear (last pool),2,0.83675,0.81900,1.775,0.106066,1.70,1.85,2
1,Full-scale conv - linear (mean pool),2,0.84450,0.82450,2.000,0.636396,1.55,2.45,2
2,Closest-budget conv - linear (report pair),2,0.83575,0.80325,3.250,0.989949,2.55,3.95,2
3,Conv d256: last - mean pooling,2,0.83675,0.84450,-0.775,0.176777,-0.90,-0.65,0
4,Linear d256: last - mean pooling,2,0.81900,0.82450,-0.550,0.707107,-1.05,-0.05,0



 Full-scale conv - linear (last pool)


,seed,a,b,gap_pp
0,30485,0.8355,0.817,1.85
1,8842,0.8380,0.821,1.70



 Full-scale conv - linear (mean pool)


,seed,a,b,gap_pp
0,30485,0.842,0.8175,2.45
1,8842,0.847,0.8315,1.55


...

,seed,a,b,gap_pp
0,30485,0.8360,0.7965,3.95
1,8842,0.8355,0.8100,2.55



 Conv d256: last - mean pooling


,seed,a,b,gap_pp
0,30485,0.8355,0.842,-0.65
1,8842,0.8380,0.847,-0.90



 Linear d256: last - mean pooling


,seed,a,b,gap_pp
0,30485,0.817,0.8175,-0.05
1,8842,0.821,0.8315,-1.05


## Experiment 1 continued — train the richer-stem main-recipe grid

The 13 configurations are trained once at seed 30485. A second independent grid is **not** required for item 1 itself; it is added separately for the noise-robustness repeat in item 3.

In [27]:
RICHER_SPECS=[]
for depth in (1,2,3,4):
    for s4_layers in (0,1,2):
        rid=f'richer_d{depth}_s4{ s4_layers }'
        RICHER_SPECS.append({
            'id':rid,
            'label':f'Richer stem depth {depth} + {s4_layers} S4D',
            'builder':lambda depth=depth,s4_layers=s4_layers: make_richer_grid_model(depth,s4_layers),
            'expected_params':RICHER_REPORT_PARAMS[(depth,s4_layers)],
            'depth':depth,'s4_layers':s4_layers,
        })
RAW_SPEC={
    'id':'richer_rawpix_s4d2',
    'label':'Richer raw-pixel S4D-only baseline',
    'builder':lambda: GalaxyClassifierS4D(s4_state=64,d_model=64,num_classes=4,colored=True),
    'expected_params':17156,'depth':0,'s4_layers':2,
}
RICHER_ALL_SPECS=RICHER_SPECS+[RAW_SPEC]

richer_checks=[]
for s in RICHER_ALL_SPECS:
    m=s['builder'](); n=sum(p.numel() for p in m.parameters())
    richer_checks.append({'id':s['id'],'actual':n,'expected':s['expected_params'],'ok':n==s['expected_params']})
display(pd.DataFrame(richer_checks))
if not all(x['ok'] for x in richer_checks):
    raise RuntimeError('Richer-grid model parameter check failed.')

richer_main_results=[]
if RUN_RICHER_GRID_MAIN:
    for spec in RICHER_ALL_SPECS:
        richer_main_results.append(train_model(spec, RICHER_GRID_MAIN_SEED, purpose='richer_grid_main_recipe'))
    richer_main_df=result_frame(richer_main_results)
    richer_main_df['stem_depth']=[next(s for s in RICHER_ALL_SPECS if s['id']==a).get('depth') for a in richer_main_df.architecture]
    richer_main_df['s4_layers']=[next(s for s in RICHER_ALL_SPECS if s['id']==a).get('s4_layers') for a in richer_main_df.architecture]
    display(richer_main_df.sort_values(['stem_depth','s4_layers']))
else:
    richer_main_df=pd.DataFrame()
    print('Richer main grid skipped.')

,id,actual,expected,ok
0,richer_d1_s40,2180,2180,True
1,richer_d1_s41,10500,10500,True
2,richer_d1_s42,18820,18820,True
3,richer_d2_s40,19844,19844,True
4,richer_d2_s41,28164,28164,True
5,richer_d2_s42,36484,36484,True
6,richer_d3_s40,29156,29156,True
7,richer_d3_s41,37476,37476,True
8,richer_d3_s42,45796,45796,True
9,richer_d4_s40,38468,38468,True


[richer_d1_s40__main__seed30485] epoch 25/630 train=0.5028 val=0.5025


[richer_d1_s40__main__seed30485] epoch 50/630 train=0.5256 val=0.5356


[richer_d1_s40__main__seed30485] epoch 75/630 train=0.5211 val=0.5150


[richer_d1_s40__main__seed30485] epoch 100/630 train=0.5370 val=0.5363


...

[richer_d1_s41__main__seed30485] epoch 150/630 train=0.7972 val=0.7806


[richer_d1_s41__main__seed30485] epoch 175/630 train=0.7794 val=0.7731


[richer_d1_s41__main__seed30485] epoch 200/630 train=0.7870 val=0.7769


[richer_d1_s41__main__seed30485] epoch 225/630 train=0.8013 val=0.7750


[richer_d1_s41__main__seed30485] epoch 250/630 train=0.8052 val=0.7744


In [ ]:
# Richer grid heatmap under the main recipe.
if len(richer_main_df):
    heat=richer_main_df[richer_main_df.stem_depth.isin([1,2,3,4])].pivot(index='stem_depth',columns='s4_layers',values='accuracy')
    fig,ax=plt.subplots(figsize=(6,4))
    im=ax.imshow(heat.values, aspect='auto')
    ax.set_xticks(range(len(heat.columns))); ax.set_xticklabels(heat.columns)
    ax.set_yticks(range(len(heat.index))); ax.set_yticklabels(heat.index)
    ax.set_xlabel('S4D layers'); ax.set_ylabel('Stem depth'); ax.set_title('Richer-stem grid — main recipe')
    for i in range(heat.shape[0]):
        for j in range(heat.shape[1]):
            ax.text(j,i,f'{heat.iloc[i,j]*100:.2f}%',ha='center',va='center')
    fig.colorbar(im,ax=ax,label='Test accuracy')
    fig.tight_layout(); fig.savefig(PLOTS_DIR/'richer_grid_main_recipe_heatmap.png',dpi=150); plt.show()

    # 0 -> 2 S4D gains at each stem depth.
    gain=(heat[2]-heat[0])*100
    print('0 -> 2 S4D gain by stem depth (pp):')
    display(gain.rename('gain_pp').to_frame())

## Experiment 3 — repeat-seed noise-robustness sweep

The report's original sweep evaluated all 13 richer-stem-grid models at test-time Gaussian noise `σ ∈ {0, 0.05, 0.1, 0.2, 0.3}`. Here the same 13 architectures are independently trained under the **main recipe** at seed 8842, then the full noise sweep is repeated. The seed-30485 grid from Experiment 1 remains the first measurement.

Because the report text does not specify whether noisy pixels were clipped to `[0,1]`, `NOISE_CLIP_TO_UNIT` is explicit rather than hidden.

In [ ]:
# ---------------------------
# Noise robustness helpers
# ---------------------------
def evaluate_with_noise(model, sigma, batch_size=64, clip_to_unit=True):
    model.eval()
    ds=TensorDataset(DATA_SPLIT['test'][0], DATA_SPLIT['test'][1])
    loader=DataLoader(ds,batch_size=batch_size,shuffle=False)
    ys, preds, probs=[],[],[]
    with torch.no_grad():
        for images,labels in loader:
            images=images.to(DEVICE)
            noisy=images + sigma*torch.randn_like(images)
            if clip_to_unit:
                noisy=noisy.clamp(0.0,1.0)
            logits=model(noisy,return_logits=True)
            p=torch.softmax(logits,dim=1)
            ys.extend(labels.numpy().tolist())
            preds.extend(torch.argmax(logits,dim=1).cpu().numpy().tolist())
            probs.extend(p.cpu().numpy().tolist())
    return macro_metrics(np.array(ys),np.array(preds),np.array(probs))


def load_trained_model(spec, seed):
    rid=run_id(spec['id'],seed)
    path=WEIGHTS_DIR/f'{rid}.pt'
    if not path.exists():
        train_model(spec,seed,purpose='dependency_train_for_noise')
    m=spec['builder']().to(DEVICE)
    m.load_state_dict(torch.load(path,map_location=DEVICE))
    return m

noise_repeat_results=[]
if RUN_NOISE_REPEAT:
    # If the main grid did not run in this session, cached seed-30485 files still work.
    for spec in RICHER_ALL_SPECS:
        noise_repeat_results.append(train_model(spec, NOISE_REPEAT_SEED, purpose='richer_grid_noise_repeat_seed'))
    noise_repeat_df=result_frame(noise_repeat_results)

    noise_rows=[]
    for spec in RICHER_ALL_SPECS:
        model=load_trained_model(spec,NOISE_REPEAT_SEED)
        for sigma in NOISE_SIGMAS:
            mm=evaluate_with_noise(model,sigma,NOISE_EVAL_BATCH_SIZE,NOISE_CLIP_TO_UNIT)
            noise_rows.append({
                'seed':NOISE_REPEAT_SEED,'architecture':spec['id'],'label':spec['label'],
                'stem_depth':spec.get('depth'),'s4_layers':spec.get('s4_layers'),'sigma':sigma,
                **{k:v for k,v in mm.items() if k!='confusion_matrix'},
            })
    noise_repeat_eval_df=pd.DataFrame(noise_rows)
    noise_repeat_eval_df.to_csv(RESULTS_DIR/'noise_repeat_seed_8842.csv',index=False)
    display(noise_repeat_eval_df.head(20))
else:
    noise_repeat_df=pd.DataFrame(); noise_repeat_eval_df=pd.DataFrame()
    print('Noise repeat suite skipped.')

In [ ]:
# Compare the two independent seeds at each sigma using the main-grid models.
# Seed 30485 evaluation is generated from its existing trained grid weights.
if RUN_NOISE_REPEAT:
    base_rows=[]
    for spec in RICHER_ALL_SPECS:
        model=load_trained_model(spec,RICHER_GRID_MAIN_SEED)
        for sigma in NOISE_SIGMAS:
            mm=evaluate_with_noise(model,sigma,NOISE_EVAL_BATCH_SIZE,NOISE_CLIP_TO_UNIT)
            base_rows.append({'seed':RICHER_GRID_MAIN_SEED,'architecture':spec['id'],'sigma':sigma,'accuracy':mm['accuracy']})
    noise_base_df=pd.DataFrame(base_rows)
    noise_all=pd.concat([noise_base_df[['seed','architecture','sigma','accuracy']],noise_repeat_eval_df[['seed','architecture','sigma','accuracy']]],ignore_index=True)
    noise_summary=(noise_all.groupby('sigma')['accuracy'].agg(['mean','std','min','max']).reset_index())
    noise_summary['mean_pct']=noise_summary['mean']*100
    noise_summary['std_pp']=noise_summary['std']*100
    display(noise_summary)

    # Plot all model curves, with separate markers for the two seeds.
    fig,ax=plt.subplots(figsize=(9,6))
    for arch,grp in noise_all.groupby('architecture'):
        for seed,sg in grp.groupby('seed'):
            ax.plot(sg['sigma'],sg['accuracy']*100,marker='o',alpha=0.55,label=f'{arch} / {seed}')
    ax.axhline(25,linestyle='--',linewidth=1,label='Random guess (25%)')
    ax.set_xlabel('Gaussian pixel noise σ'); ax.set_ylabel('Test accuracy (%)')
    ax.set_title('Richer-stem noise robustness — repeat seeds')
    # Avoid a giant legend: place it outside only when manageable.
    ax.legend(fontsize=6,ncol=2,bbox_to_anchor=(1.02,1),loc='upper left')
    fig.tight_layout(); fig.savefig(PLOTS_DIR/'noise_robustness_repeat_seeds.png',dpi=150,bbox_inches='tight'); plt.show()

## Experiment 4 — GroupNorm portability / folding test

A conventional BatchNorm can be folded exactly because its inference statistics are fixed. `GroupNorm` is different: its mean and variance are computed from the current sample, so exact static folding into the preceding convolution is not possible.

The practical portability experiment here is therefore:

1. Train the richer-stem `depth=4 + 2 S4D` model under the main recipe.
2. Using **training-set activations only**, estimate fixed mean/variance for every GroupNorm group.
3. Replace each GroupNorm with the corresponding **fixed affine transform** and absorb that affine transform into the preceding convolution's weight/bias.
4. Evaluate the original and folded checkpoints on validation/test data with no fine-tuning.

The result answers the concrete portability question: *how much of the CNNStem advantage survives when dynamic GroupNorm is removed?* It does **not** claim exact mathematical equivalence to the original GroupNorm model.

In [ ]:
# ---------------------------
# Calibrated fixed-GroupNorm folding
# ---------------------------

def _gn_stats_from_input(x, gn):
    B,C,H,W=x.shape
    G=gn.num_groups
    y=x.reshape(B,G,C//G,H,W)
    mean=y.mean(dim=(2,3,4))
    var=y.var(dim=(2,3,4),unbiased=False)
    return mean, var


def collect_gn_calibration(model, max_batches=None):
    model.eval()
    stem=model.cnn_stem
    sums={}; sums2={}; counts={}
    hooks=[]
    for name,mod in stem.named_modules():
        if isinstance(mod,nn.GroupNorm):
            sums[name]=None; sums2[name]=None; counts[name]=0
            def hook(mod, inp, out, name=name):
                x=inp[0].detach()
                m,v=_gn_stats_from_input(x,mod)
                n=m.shape[0]
                sm=m.sum(dim=0)
                sm2=(v+m*m).sum(dim=0)
                if sums[name] is None:
                    sums[name]=sm.clone(); sums2[name]=sm2.clone()
                else:
                    sums[name]+=sm; sums2[name]+=sm2
                counts[name]+=n
            hooks.append(mod.register_forward_hook(hook))
    loader=DataLoader(TensorDataset(DATA_SPLIT['train'][0],DATA_SPLIT['train'][1]),batch_size=64,shuffle=False)
    with torch.no_grad():
        for bi,(images,_) in enumerate(loader):
            _=model(images.to(DEVICE),return_logits=True)
            if max_batches is not None and bi+1>=max_batches:
                break
    for h in hooks: h.remove()
    stats={}
    for name in sums:
        mean=sums[name]/counts[name]
        second=sums2[name]/counts[name]
        var=(second-mean*mean).clamp_min(0.0)
        stats[name]={'mean':mean.cpu(),'var':var.cpu(),'count':counts[name]}
    return stats


def _fold_conv_with_group_stats(conv, gn, mean_g, var_g):
    # Fixed GN on a channel c belonging to group g is: gamma_c*(x_c-mu_g)/sqrt(var_g+eps)+beta_c.
    # This becomes a channel-wise affine transform, which can be absorbed into Conv2d.
    C=conv.out_channels
    G=gn.num_groups
    group_size=C//G
    device=conv.weight.device
    mean_c=mean_g.to(device).repeat_interleave(group_size)
    var_c=var_g.to(device).repeat_interleave(group_size)
    scale=gn.weight.to(device)/torch.sqrt(var_c+gn.eps)
    shift=gn.bias.to(device)-scale*mean_c
    with torch.no_grad():
        w=conv.weight.data
        conv.weight.data=w*scale.view(-1,1,1,1)
        if conv.bias is None:
            conv.bias=nn.Parameter(shift.clone())
        else:
            conv.bias.data=conv.bias.data*scale+shift


def fold_richer_stem_inplace(model, stats):
    stem=model.cnn_stem
    # Fold only GN instances immediately after convolutions.
    pairs=[]
    for conv_name,gn_name in [
        ('conv1','norm1'),('conv2','norm2'),('conv3','norm3'),('res_conv','res_norm')
    ]:
        if hasattr(stem,conv_name) and hasattr(stem,gn_name):
            conv=getattr(stem,conv_name); gn=getattr(stem,gn_name)
            pairs.append((conv_name,gn_name,conv,gn))
    for conv_name,gn_name,conv,gn in pairs:
        s=stats[gn_name]
        _fold_conv_with_group_stats(conv,gn,s['mean'],s['var'])
        setattr(stem,gn_name,nn.Identity())
    return model


gn_spec=next(s for s in RICHER_SPECS if s['id']=='richer_d4_s42')
if RUN_GN_FOLD:
    # Train/reuse the exact model that also appears in Experiment 1.
    if not (WEIGHTS_DIR / f'{run_id(gn_spec["id"], GN_FOLD_SEED)}.pt').exists():
        train_model(gn_spec,GN_FOLD_SEED,purpose='gn_fold_dependency')
    original=gn_spec['builder']().to(DEVICE)
    original.load_state_dict(torch.load(WEIGHTS_DIR/f'{run_id(gn_spec["id"],GN_FOLD_SEED)}.pt',map_location=DEVICE))
    stats=collect_gn_calibration(original)
    folded=copy.deepcopy(original)
    fold_richer_stem_inplace(folded,stats)
    folded.to(DEVICE).eval()

    val_loader=DataLoader(TensorDataset(DATA_SPLIT['val'][0],DATA_SPLIT['val'][1]),batch_size=64,shuffle=False)
    test_loader=DataLoader(TensorDataset(DATA_SPLIT['test'][0],DATA_SPLIT['test'][1]),batch_size=64,shuffle=False)
    orig_val=evaluate_model(original,val_loader); folded_val=evaluate_model(folded,val_loader)
    orig_test=evaluate_model(original,test_loader); folded_test=evaluate_model(folded,test_loader)
    fold_summary=pd.DataFrame([
        {'model':'Original GroupNorm','split':'val',**{k:v for k,v in orig_val.items() if k!='confusion_matrix'}},
        {'model':'Calibrated fixed-GN folded','split':'val',**{k:v for k,v in folded_val.items() if k!='confusion_matrix'}},
        {'model':'Original GroupNorm','split':'test',**{k:v for k,v in orig_test.items() if k!='confusion_matrix'}},
        {'model':'Calibrated fixed-GN folded','split':'test',**{k:v for k,v in folded_test.items() if k!='confusion_matrix'}},
    ])
    display(fold_summary)
    print('Test accuracy change after fold (pp):', (folded_test['accuracy']-orig_test['accuracy'])*100)

    # Portability artifact: save the folded state dict and calibration stats.
    torch.save(folded.state_dict(),EXPORT_DIR/'richer_d4_s42_gn_folded_state.pt')
    serializable={k:{'mean':v['mean'].tolist(),'var':v['var'].tolist(),'count':v['count']} for k,v in stats.items()}
    (EXPORT_DIR/'richer_d4_s42_gn_calibration.json').write_text(json.dumps(serializable,indent=2))
else:
    fold_summary=pd.DataFrame()
    print('GN folding test skipped.')

## Final consolidated report

This cell creates compact CSV/JSON artifacts that can be downloaded from Kaggle after the run. It deliberately keeps **reported historical numbers** separate from the new measurements generated by this notebook.

In [ ]:
# ---------------------------
# Consolidate outputs
# ---------------------------
manifest={
    'notebook':'s4d_future_work_validation',
    'split_seed':SPLIT_SEED,
    'main_recipe':MAIN_RECIPE,
    'production_seeds':PRODUCTION_SEEDS,
    'richer_grid_main_seed':RICHER_GRID_MAIN_SEED,
    'noise_repeat_seed':NOISE_REPEAT_SEED,
    'noise_sigmas':NOISE_SIGMAS,
    'noise_clip_to_unit':NOISE_CLIP_TO_UNIT,
    'run_flags':{
        'RUN_RICHER_GRID_MAIN':RUN_RICHER_GRID_MAIN,
        'RUN_PRODUCTION_REPEATS':RUN_PRODUCTION_REPEATS,
        'RUN_NOISE_REPEAT':RUN_NOISE_REPEAT,
        'RUN_GN_FOLD':RUN_GN_FOLD,
        'RUN_STRICT_NEAR_MATCH_BUDGET_PAIR':RUN_STRICT_NEAR_MATCH_BUDGET_PAIR,
    },
    'historical_anchors_from_report':{
        'production_best_s4d_retrain_seed30485_main':0.8530,
        'richer_hybrid_55k_2l_seed30485_main':0.8975,
        'production_noise_floor_pp':3.25,
        'richer_specific_gap_noise_range_pp':[0.10,0.65],
    },
}
(RESULTS_DIR/'manifest.json').write_text(json.dumps(manifest,indent=2))

if len(production_df):
    production_df.to_csv(RESULTS_DIR/'production_repeat_results.csv',index=False)
if len(comparison_df):
    comparison_df.to_csv(RESULTS_DIR/'production_comparison_summary.csv',index=False)
if len(richer_main_df):
    richer_main_df.to_csv(RESULTS_DIR/'richer_grid_main_results.csv',index=False)
if len(fold_summary):
    fold_summary.to_csv(RESULTS_DIR/'gn_fold_summary.csv',index=False)

# Zip everything except model checkpoints unless explicitly requested; weights remain separate so a results
# download stays small. Kaggle lets you download the whole working directory too.
import zipfile
results_zip=BASE_DIR/'s4d_future_work_results.zip'
with zipfile.ZipFile(results_zip,'w',zipfile.ZIP_DEFLATED) as z:
    for p in RESULTS_DIR.rglob('*'):
        if p.is_file(): z.write(p,p.relative_to(BASE_DIR))
    for p in PLOTS_DIR.rglob('*'):
        if p.is_file(): z.write(p,p.relative_to(BASE_DIR))
print('Results archive:',results_zip)
print('Weights directory:',WEIGHTS_DIR)
print('Export directory:',EXPORT_DIR)

### Interpretation checklist

When writing the final report update, use the repeated-seed results rather than single-run point estimates. For each paired production comparison, report the per-seed gaps and their mean/spread; for the richer grid, report the 12 cell values and the `0→2 S4D` gains; for noise robustness, compare the two-seed curves and especially the `σ=0.30` survival range; for GroupNorm folding, report the original-versus-folded test gap and whether the folded model remains above the report's 85.30% production-family main-recipe retrain anchor.

A failure of the fixed-statistics fold should be interpreted as evidence that **dynamic normalization matters**, not as evidence that the CNNStem itself is non-portable. Conversely, a small fold loss would support replacing GroupNorm with an export-time fixed affine approximation, but would still require a separate bare-metal implementation test.